# ERPA — build a complete custom agent harness

ERPA is an enterprise retail-planning assistant for Alex, a regional merchandiser responsible for inventory, sales performance, competitor signals, and daily operating communications. A useful answer must do more than sound plausible: it must query current Oracle data, apply governed business definitions, remember prior decisions, retrieve trusted procedures and tools, isolate generated code, checkpoint control flow, and leave an observable execution trace.

This notebook builds that complete harness from first principles. It imports no course-local application package and hides no setup behind an appbook. Every important boundary is visible and executable: an Oracle AI Database container and schema, deterministic retail fixtures, an in-database embedding model, a living semantic layer, a SecureFile scratch filesystem, Oracle Agent Memory, progressive skill and tool disclosure, an E2B computation boundary, a LangGraph loop, Oracle checkpoints, LangSmith tracing, a semantic cache, durable human approval, Oracle Scheduler automation, acceptance tests, and scoped cleanup.

The central scenario is Alex's morning brief. ERPA must:

1. find stock below reorder point;
2. suppress issues already covered by an open purchase order;
3. explain unusual sales movement without turning correlation into causation;
4. use the governed definition of profitability;
5. remember an explicit preference across a fresh session; and
6. prove which database, tool, memory, checkpoint, and trace supported the answer.

That scenario is deliberately small enough to inspect while still exposing the concerns that make a production agent different from a chatbot.

> **Learning contract**
>
> By the end, you should be able to point to the exact code and Oracle object responsible for each kind of state: business facts, semantic meaning, short-term working files, durable agent memory, graph checkpoints, cache entries, approvals, schedules, and traces. You should also be able to explain why those state classes are not interchangeable.

## How to run the notebook

Run the cells in order. The notebook is rerunnable: creation helpers tolerate existing objects, fixture cells restore deterministic source data, jobs are replaced explicitly, and the final cleanup is opt-in. Live execution requires Docker, an Anthropic key, an E2B key, and a LangSmith key. Secret prompts use `getpass`, so values do not appear in cell output or the saved notebook.

The assertions are part of the lesson. They test architectural claims instead of exact model prose: three uncovered restock actions, one open-PO suppression, a WarmLayer spike without an internal promotion, UK leadership under the governed gross-margin definition, E2B-only generated code, durable scratch promotion, semantic-cache bypass, resumable checkpoints, durable human approval, and visible LangSmith runs.

## The harness at a glance

An agent harness is the runtime around the model. The model proposes the next step; the harness decides what context is admissible, which tools can run, where generated code executes, what state survives, when a person must approve an effect, and what evidence is recorded.

```mermaid
flowchart LR
    U[Alex or Scheduler] --> C{Semantic cache}
    C -->|eligible hit| R[Return cached generation]
    C -->|miss or bypass| G[LangGraph control loop]
    G --> A[Assemble selective context]
    SL[(Living semantic layer)] --> A
    OAMP[(Oracle Agent Memory)] --> A
    SF[(SecureFile ScratchFS)] --> A
    A --> M[Claude Opus 4.8\nadaptive thinking]
    M --> T{Allowlisted tool call?}
    T -->|trusted SQL/retrieval| DB[(Oracle AI Database)]
    T -->|generated Python| E[E2B sandbox]
    DB --> A
    E --> SF
    A --> P[Persist completed turn]
    P --> OAMP
    G --> CK[(OracleSaver checkpoints)]
    G --> LS[LangSmith traces]
    G --> H{LangGraph interrupt}
    H -->|human resume command| X[Approved adapter]
    P --> C
```

The arrows reveal five distinct boundaries:

- **Authority boundary.** Oracle views, constraints, comments, and curated hints define what terms such as “profitable” mean. The language model is not allowed to invent a metric.
- **Execution boundary.** Trusted adapters run allowlisted SQL and filesystem operations. Model-generated Python runs only in E2B; the notebook provides no host fallback.
- **State boundary.** Scratch files, checkpoints, conversation messages, semantic memories, cache entries, and business rows each have different owners and lifetimes.
- **Effect boundary.** A model request is not approval. LangGraph pauses with `interrupt()` and resumes only when the host supplies a `Command(resume=...)` decision under the same checkpoint thread.
- **Evidence boundary.** LangSmith spans, Oracle audit rows, cache counters, scheduler rows, and assertions make behavior inspectable after the answer is produced.

Oracle is the unifying data substrate, but unification does not erase semantics. A checkpoint exists so a computation can resume; a memory exists so a future conversation can recall a fact; a scratch file exists so the current session can externalize working state; a cache entry exists to avoid repeated computation; and a business table remains the source of operational truth.

Prompt caching is discussed later as a complementary optimization, but it is intentionally not enabled. Semantic caching of completed answers is implemented and tested in Oracle.

## Custom harness component map

This table is the working vocabulary for the custom build. Unlike a framework package, a custom harness has no hidden defaults: every **Built** row has concrete code in this notebook. **Partial** means a useful seam exists but production policy or lifecycle work remains. **Missing** means the capability is intentionally visible as a gap rather than implied.

| Harness concern | Custom component | Role in ERPA | Status |
|---|---|---|---|
| Agent loop | LangGraph `StateGraph` | Context → Claude → tools → persistence under explicit budgets | **Built** |
| Construction | Notebook factories + typed `AgentState` | Wires model, memory, toolbox, skills, cache, tracing and checkpoints | **Built; custom code** |
| Model adapter | `ChatAnthropic` / Claude Opus 4.8 | Adaptive-thinking reasoning and tool selection | **Built; Anthropic only** |
| Persona | `SYSTEM` instruction | Stable ERPA role, authority and safety policy | **Partial; no persona lifecycle/evolution** |
| Application modes | — | Preconfigured assistant/workflow/research bundles | **Missing** |
| Database substrate | `python-oracledb` + Oracle AI Database Free | One schema for business data, vectors, jobs, scratch, cache and checkpoints | **Built** |
| Conversation memory | OAMP thread messages | User/assistant episodic record by thread | **Built** |
| Knowledge memory | OAMP facts/guidelines + institutional-doc table | Durable business rules and retrieved institutional evidence | **Built** |
| Entity memory | OAMP facts + relational product/region tables | Facts about Alex, products, regions and purchase orders | **Partial; no dedicated entity API** |
| Working memory | OAMP context card + SecureFile `ScratchFS` | Bounded recent context, current plan, notes and artifacts | **Built** |
| Summaries | OAMP context summaries/cards | Compresses conversation into prompt-ready working memory | **Built; source-link audit not implemented** |
| Semantic cache | `OracleSemanticCache` from `langchain-oracledb` | Returns meaning-equivalent read-only answers before the graph | **Built in Part 11** |
| Function tools | `CustomToolbox` + LangChain `StructuredTool` | Semantic discovery, progressive schema disclosure, validation and allowlisted execution | **Built** |
| Tool-result compaction | E2B result → ScratchFS pointer | Keeps large generated-code output out of model context | **Partial; only E2B output is compacted** |
| Skills | Oracle `ERPA_SKILL_REGISTRY` | Semantically retrieves a short manifest and loads one full procedure on demand | **Built; curated ACTIVE skills only** |
| Continual learning | Scratch → trigger → promotion queue → OAMP | Promotes selected session notes into durable memory | **Partial; no workflow-to-skill harvester/shadow review** |
| MCP | — | External standardized tool/resource transports | **Missing** |
| Code sandbox | E2B Code Interpreter | Executes model-generated Python with no host fallback | **Built** |
| Multi-agent orchestration | — | Delegate bounded work to specialists and consolidate it | **Missing; this is a single-agent graph** |
| Automations | `DBMS_SCHEDULER` + queue worker | Enqueues a weekday brief and runs it through the same graph | **Built** |
| Internet access | Governed external-signal ingestion table | Supplies dated outside evidence | **Partial; no live search connector** |
| Self-awareness | — | Guarded inspection or mutation of harness source | **Missing** |
| Context telemetry | LangSmith traces + cache/model/tool counters | Shows graph spans and proves cache bypass | **Partial; no token-window dashboard** |
| Human in the loop | LangGraph `interrupt()` + `Command(resume=...)` + Oracle audit | Pauses durably and binds a human decision to one concrete effect | **Built; adapter is a demonstration** |
| Semantic layer | Views, comments, hints, foreign keys, `V$SQL`, vector catalog | Grounds metrics, joins and living workload meaning | **Built** |
| Graph checkpoints | `OracleSaver` | Persists every graph super-step under `thread_id` | **Built** |
| Prompt caching | Stable-prefix design note | Future Anthropic input-token optimization | **Not implemented, by design** |

The missing rows are not required for this ERPA scenario. They are the clearest places where a reusable framework can provide leverage over a deliberately explicit custom build.


### Reading the map

“Built” means this notebook contains executable code and an assertion for the capability. It does not mean the teaching implementation has every production control. “Partial” identifies the exact boundary that remains. “Missing” is equally important: a custom harness should never imply that multi-agent delegation, MCP transport, live web ingestion, or source self-modification exists when it does not.

The map is also a debugging index. If an answer is factually wrong, inspect the source tables, semantic view, or retrieval evidence. If a run cannot resume, inspect the OracleSaver thread. If the agent forgets an accepted preference, inspect OAMP rather than the graph checkpoint. If a response is unexpectedly reused, inspect cache eligibility, namespace, threshold, and data-version fingerprint. This separation prevents one component from silently compensating for another.

# Part 1 · Reproducible environment

The environment cell is an executable bill of materials. Versions are constrained at the public package boundary instead of using unpublished Git commit archives. In particular, the Oracle integrations come from the latest released `langchain-oracledb` and `langgraph-oracledb` packages so their documented cache and checkpoint APIs are reproducible.

| Package group | Responsibility |
|---|---|
| Anthropic + LangChain | Claude Opus 4.8 adapter, messages, tools, and generations |
| LangGraph + Oracle saver | Explicit control graph and durable checkpoint state |
| `langchain-oracledb` | Oracle semantic cache and distance strategy |
| Oracle Agent Memory | Threads, context cards, semantic facts, preferences, and hybrid retrieval |
| `python-oracledb` | Thin-mode connections, pools, vectors, LOBs, PL/SQL, and Scheduler |
| E2B | Remote isolation for generated Python |
| LangSmith | Trace trees for model, graph, retriever, memory, and tool spans |

Run the install cell once. If the kernel reports that an already-imported package changed, restart the kernel before continuing so Python does not mix old modules with newly installed distributions.

In [1]:
%pip install -U \
  "anthropic>=0.117" \
  "langchain-anthropic>=1.5" \
  "langchain-core>=1.2" \
  "langgraph>=1.2,<2" \
  "langchain-oracledb==1.5.0" \
  "langgraph-oracledb==1.0.1" \
  "oracleagentmemory[litellm]==26.6.0" \
  "oracledb>=4.0.2" \
  "langsmith>=0.4" \
  "e2b-code-interpreter==2.9.0" \
  "e2b==2.37.1" \
  "requests>=2.32" \
  "pandas>=2.2" \
  "transformers>=4.41,<5" \
  "ipywidgets>=8" \
  "packaging>=24"


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Configuration and secret handling

Secrets are gathered with `getpass`, never hard-coded and never printed. `require_secret` first accepts a pre-seeded environment variable for unattended CI or notebook execution; otherwise it opens a masked prompt. An empty answer fails immediately instead of producing a confusing authentication error several parts later.

The requested credentials are:

| Variable | Used by | Why it is required |
|---|---|---|
| `ORACLE_ADMIN_PASSWORD` | SYS connection | Creates/grants the isolated workshop schema |
| `ANTHROPIC_API_KEY` | `ChatAnthropic` and OAMP extraction | Runs Opus 4.8 reasoning and memory extraction |
| `E2B_API_KEY` | E2B Code Interpreter | Executes generated Python outside the host |
| `LANGSMITH_API_KEY` | LangSmith client/tracing | Records and verifies execution traces |

The Oracle application password is configuration for the disposable workshop schema. In production it should also come from a secret manager. The DSN parser derives the Docker port from `ORA_DSN` unless `ERPA_ORACLE_PORT` is explicitly set; that prevents the container lifecycle check from probing a different port than the database client.

Only non-sensitive diagnostics are printed: DSN, schema name, model, thinking mode, project, and a Boolean stating that required credentials are present. If a LangSmith PAT can access multiple workspaces, set `LANGSMITH_WORKSPACE_ID`; the ID is routing metadata, not an API key.

In [2]:
import base64
import hashlib
import json
import logging
import os
import re
import shutil
import socket
import subprocess
import tempfile
import time
import uuid
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path, PurePosixPath
from typing import Any, Dict, List, Optional, TypedDict, Union

import oracledb
import pandas as pd
import requests

oracledb.defaults.fetch_lobs = False

# LangSmith's background uploader can otherwise emit one warning per span when
# an external token/workspace is unavailable. Part 15 reports that status once.
logging.getLogger("langsmith.client").setLevel(logging.CRITICAL)


def require_secret(name: str, prompt: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        value = getpass(prompt).strip()
    if not value:
        raise RuntimeError(f"{name} is required for the complete live workshop.")
    os.environ[name] = value
    return value


DEFAULT_ORACLE_DSN = os.getenv("ORA_DSN", "127.0.0.1:1523/FREEPDB1")


def dsn_port(dsn: str, default: int = 1523) -> int:
    match = re.search(r":(\d+)(?:/|$)", dsn)
    return int(match.group(1)) if match else default


@dataclass(frozen=True)
class Config:
    oracle_image: str = os.getenv("ERPA_ORACLE_IMAGE", "container-registry.oracle.com/database/free:latest-lite")
    oracle_container: str = os.getenv("ERPA_ORACLE_CONTAINER", "erpa-custom-oracle-26ai")
    oracle_port: int = int(os.getenv("ERPA_ORACLE_PORT", str(dsn_port(DEFAULT_ORACLE_DSN))))
    oracle_dsn: str = DEFAULT_ORACLE_DSN
    oracle_user: str = os.getenv("ORA_AGENT_USER", "ERPA_AGENT")
    oracle_tablespace: str = os.getenv("ORA_AGENT_TABLESPACE", "ERPA_DATA")
    oracle_password: str = os.getenv("ORA_AGENT_PWD", "ErpaAgent_2026!")
    model: str = os.getenv("ANTHROPIC_MODEL", "claude-opus-4-8")
    embed_model: str = os.getenv("ERPA_EMBED_MODEL", "ALL_MINILM_L12_V2")
    embed_dim: int = 384
    user_id: str = os.getenv("ERPA_USER_ID", "alex-course-user")
    agent_id: str = "erpa-custom-agent"
    tenant_id: str = "erpa-workshop"
    memory_store_id: str = "ERPACUSTOM"
    langsmith_project: str = os.getenv("LANGSMITH_PROJECT", "erpa-custom-harness")
    keep_data: bool = os.getenv("ERPA_KEEP_DATA", "1") == "1"


CFG = Config()
ORACLE_ADMIN_PASSWORD = require_secret("ORACLE_ADMIN_PASSWORD", "Oracle SYS password: ")
ANTHROPIC_API_KEY = require_secret("ANTHROPIC_API_KEY", "Anthropic API key: ")
E2B_API_KEY = require_secret("E2B_API_KEY", "E2B API key: ")
LANGSMITH_API_KEY = require_secret("LANGSMITH_API_KEY", "LangSmith API key: ")

os.environ.update({
    "ANTHROPIC_API_KEY": ANTHROPIC_API_KEY,
    "E2B_API_KEY": E2B_API_KEY,
    "LANGSMITH_API_KEY": LANGSMITH_API_KEY,
    "LANGSMITH_TRACING": "true",
    "LANGCHAIN_TRACING_V2": "true",
    "LANGSMITH_PROJECT": CFG.langsmith_project,
})

print({
    "oracle_dsn": CFG.oracle_dsn,
    "oracle_user": CFG.oracle_user,
    "model": CFG.model,
    "thinking": "adaptive",
    "langsmith_project": CFG.langsmith_project,
    "required_credentials_present": all((ANTHROPIC_API_KEY, E2B_API_KEY, LANGSMITH_API_KEY)),
})

{'oracle_dsn': '127.0.0.1:1523/FREEPDB1', 'oracle_user': 'ERPA_AGENT', 'model': 'claude-opus-4-8', 'thinking': 'adaptive', 'langsmith_project': 'erpa-custom-harness', 'required_credentials_present': True}


## Start or reuse Oracle AI Database

The notebook owns the database-container setup rather than depending on a repository Compose file. The lifecycle helper follows an idempotent decision tree:

1. if the configured DSN already accepts a SYS connection, reuse it;
2. otherwise, if the named container exists but is stopped, start it;
3. otherwise, create the Oracle AI Database Free container with an explicit port and admin password;
4. poll both the TCP listener and `FREEPDB1` authentication until the database is genuinely ready.

A listening socket alone is not readiness evidence: Oracle may accept TCP before the pluggable database can authenticate. The retry loop therefore proves an actual SYS connection. Transient listener/startup errors are retried with visible progress, but invalid credentials and an exited Docker container fail immediately with an actionable diagnostic instead of waiting eight minutes. No secret is included in the printed diagnostics.

> **Production note:** container startup is a workshop convenience. A deployed harness should receive a managed Oracle service DSN and scoped application credential; schema migration belongs in a controlled deployment pipeline.

In [3]:
def port_is_open(host: str = "127.0.0.1", port: int = CFG.oracle_port) -> bool:
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False


def docker_state(container: str) -> Optional[str]:
    if not shutil.which("docker"):
        return None
    result = subprocess.run(
        ["docker", "inspect", "--format", "{{.State.Status}}", container],
        capture_output=True, text=True,
    )
    return result.stdout.strip() if result.returncode == 0 else None


def docker_log_tail(container: str, lines: int = 30) -> str:
    result = subprocess.run(
        ["docker", "logs", "--tail", str(lines), container],
        capture_output=True, text=True,
    )
    return (result.stdout + result.stderr).strip()[-4000:]


def docker_published_ports(container: str, container_port: str = "1521/tcp") -> set[int]:
    result = subprocess.run(
        ["docker", "port", container, container_port],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        return set()
    return {int(match.group(1)) for line in result.stdout.splitlines()
            if (match := re.search(r":(\d+)\s*$", line))}


def oracle_error_code(exc: oracledb.Error) -> str:
    detail = exc.args[0] if exc.args else None
    return str(getattr(detail, "full_code", "") or "")


if not port_is_open():
    if not shutil.which("docker"):
        raise RuntimeError("Docker is unavailable and no Oracle listener is reachable.")
    inspected = subprocess.run(
        ["docker", "inspect", CFG.oracle_container], capture_output=True, text=True
    )
    if inspected.returncode == 0:
        subprocess.run(
            ["docker", "start", CFG.oracle_container],
            check=True, capture_output=True, text=True,
        )
        print(f"Started Oracle container {CFG.oracle_container}; waiting for FREEPDB1.")
    else:
        docker_env = {**os.environ, "ORACLE_PWD": ORACLE_ADMIN_PASSWORD}
        subprocess.run([
            "docker", "run", "-d", "--platform", "linux/amd64",
            "--name", CFG.oracle_container,
            "-p", f"{CFG.oracle_port}:1521", "-e", "ORACLE_PWD",
            "-v", "erpa-custom-oracle-data:/opt/oracle/oradata",
            CFG.oracle_image,
        ], check=True, env=docker_env, capture_output=True, text=True)
        print(f"Created Oracle container {CFG.oracle_container}; first startup can take several minutes.")
    published_ports = docker_published_ports(CFG.oracle_container)
    if CFG.oracle_port not in published_ports:
        actual = ", ".join(str(port) for port in sorted(published_ports)) or "none"
        raise RuntimeError(
            f"Oracle endpoint mismatch: CFG.oracle_dsn is {CFG.oracle_dsn}, but Docker "
            f"container {CFG.oracle_container} publishes 1521 on host port(s) {actual}. "
            "An existing container keeps its original port mapping when restarted. Clear a "
            "stale ORA_DSN/ERPA_ORACLE_PORT override or set ORA_DSN to "
            f"127.0.0.1:{min(published_ports) if published_ports else 1523}/FREEPDB1, "
            "then rerun the configuration cell."
        )

started_waiting = time.monotonic()
deadline = started_waiting + 480
next_progress = started_waiting
last_error = None
while time.monotonic() < deadline:
    try:
        with oracledb.connect(
            user="sys", password=ORACLE_ADMIN_PASSWORD, dsn=CFG.oracle_dsn,
            mode=oracledb.AUTH_MODE_SYSDBA,
        ) as _ready:
            break
    except oracledb.Error as exc:
        last_error = exc
        code = oracle_error_code(exc)
        if code == "ORA-01017":
            raise RuntimeError(
                f"SYS authentication failed at {CFG.oracle_dsn}. Stop this cell, rerun the "
                "secret/configuration cell, and enter the ORACLE_PWD used when this container "
                "was created. The existing erpa-custom-oracle-26ai workshop container uses "
                "the ERPA application workshop password, not the separate port-1522 "
                "MemoRizz database password."
            ) from exc
        state = docker_state(CFG.oracle_container)
        if state in {"exited", "dead"}:
            raise RuntimeError(
                f"Oracle container {CFG.oracle_container} stopped during startup.\n"
                f"Recent container log:\n{docker_log_tail(CFG.oracle_container)}"
            ) from exc
        now = time.monotonic()
        if now >= next_progress:
            elapsed = int(now - started_waiting)
            print(f"Waiting for Oracle FREEPDB1: {elapsed}s; last status={code or type(exc).__name__}.")
            next_progress = now + 15
        time.sleep(3)
else:
    raise TimeoutError(f"Oracle did not become ready within eight minutes: {last_error}")

print(f"Oracle is accepting connections at {CFG.oracle_dsn}.")


Oracle is accepting connections at 127.0.0.1:1523/FREEPDB1.


## Create the ERPA schema and load an in-database embedder

The notebook uses two privilege levels for different purposes. SYS is used only to create the isolated schema, its dedicated `ERPA_DATA` tablespace, and the small set of capabilities needed by the lab. All business objects, queries, memory stores, vectors, jobs, scratch files, checkpoints, and cache entries are then owned through the `ERPA_AGENT` application connection.

> **Why a dedicated tablespace?** Oracle's `VECTOR` type requires automatic segment space management (ASSM). The Free container's initial default permanent tablespace is `SYSTEM`, which uses manual segment management and raises `ORA-43853` for vector columns. The bootstrap therefore creates `ERPA_DATA` with `SEGMENT SPACE MANAGEMENT AUTO`, assigns it even when `ERPA_AGENT` already exists, and grants quota only on that application tablespace.

The embedding model is loaded as an ONNX mining model and called with Oracle's `VECTOR_EMBEDDING` SQL function. That creates one consistent 384-dimensional vector space for the living semantic catalog, institutional document retrieval, tool/skill discovery, Oracle Agent Memory, and semantic cache. Keeping embeddings in the database avoids exporting every document merely to compute or compare a vector.

The helper functions establish important transaction rules:

- `fetch_all` returns named dictionaries and closes only connections it owns;
- `execute` commits trusted writes explicitly;
- `ddl` ignores only named idempotency errors and re-raises everything else;
- schema and password identifiers are validated before being interpolated into unavoidable DDL.

The final smoke query asserts the model's output dimension. If that assertion fails, all downstream vector tables would have an incompatible shape, so the notebook stops here.

In [4]:
if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_$#]{0,29}", CFG.oracle_user):
    raise ValueError("ORA_AGENT_USER must be a safe unquoted Oracle identifier")
if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_$#]{0,29}", CFG.oracle_tablespace):
    raise ValueError("ORA_AGENT_TABLESPACE must be a safe unquoted Oracle identifier")
if not re.fullmatch(r"[A-Za-z0-9_#$!.-]{12,64}", CFG.oracle_password):
    raise ValueError("ORA_AGENT_PWD must be a 12–64 character workshop password")


def connect(user: Optional[str] = None, password: Optional[str] = None, *, sysdba: bool = False):
    return oracledb.connect(
        user=user or CFG.oracle_user,
        password=password or CFG.oracle_password,
        dsn=CFG.oracle_dsn,
        mode=oracledb.AUTH_MODE_SYSDBA if sysdba else oracledb.AUTH_MODE_DEFAULT,
    )


with connect("sys", ORACLE_ADMIN_PASSWORD, sysdba=True) as admin:
    cur = admin.cursor()
    cur.execute(
        "SELECT segment_space_management FROM dba_tablespaces WHERE tablespace_name=:name",
        {"name": CFG.oracle_tablespace.upper()},
    )
    tablespace_row = cur.fetchone()
    if tablespace_row is None:
        cur.execute(
            "SELECT file_name FROM dba_data_files "
            "WHERE tablespace_name='SYSTEM' FETCH FIRST 1 ROW ONLY"
        )
        system_datafile = cur.fetchone()[0]
        app_datafile = (
            Path(system_datafile).parent / f"{CFG.oracle_tablespace.lower()}01.dbf"
        ).as_posix().replace("'", "''")
        cur.execute(
            f"CREATE TABLESPACE {CFG.oracle_tablespace} "
            f"DATAFILE '{app_datafile}' SIZE 512M "
            "AUTOEXTEND ON NEXT 128M MAXSIZE 8G "
            "EXTENT MANAGEMENT LOCAL SEGMENT SPACE MANAGEMENT AUTO"
        )
    elif tablespace_row[0] != "AUTO":
        raise RuntimeError(
            f"{CFG.oracle_tablespace} must use automatic segment space management"
        )
    try:
        cur.execute(
            f'CREATE USER {CFG.oracle_user} IDENTIFIED BY "{CFG.oracle_password}" '
            f'DEFAULT TABLESPACE {CFG.oracle_tablespace} TEMPORARY TABLESPACE TEMP '
            f'QUOTA UNLIMITED ON {CFG.oracle_tablespace}'
        )
    except oracledb.DatabaseError as exc:
        if "ORA-01920" not in str(exc):
            raise
    cur.execute(
        f"ALTER USER {CFG.oracle_user} DEFAULT TABLESPACE {CFG.oracle_tablespace} "
        "TEMPORARY TABLESPACE TEMP"
    )
    cur.execute(
        f"ALTER USER {CFG.oracle_user} QUOTA UNLIMITED ON {CFG.oracle_tablespace}"
    )
    grants = [
        "CREATE SESSION", "CREATE TABLE", "CREATE VIEW", "CREATE PROCEDURE",
        "CREATE SEQUENCE", "CREATE TRIGGER", "CREATE JOB", "CREATE MINING MODEL",
        "CREATE DOMAIN", "SELECT_CATALOG_ROLE",
    ]
    for grant in grants:
        try:
            cur.execute(f"GRANT {grant} TO {CFG.oracle_user}")
        except oracledb.DatabaseError as exc:
            print("Grant warning:", grant, str(exc).splitlines()[0])
    for statement in (
        f"GRANT EXECUTE ON DBMS_SCHEDULER TO {CFG.oracle_user}",
        f"GRANT EXECUTE ON DBMS_VECTOR TO {CFG.oracle_user}",
        f"GRANT EXECUTE ON DBMS_VECTOR_CHAIN TO {CFG.oracle_user}",
        f"GRANT SELECT ON SYS.V_$SQL TO {CFG.oracle_user}",
    ):
        try:
            cur.execute(statement)
        except oracledb.DatabaseError as exc:
            print("Grant warning:", str(exc).splitlines()[0])
    admin.commit()
print(
    f"{CFG.oracle_user} default tablespace: {CFG.oracle_tablespace} "
    "(automatic segment space management)."
)


def fetch_all(sql: str, binds: Optional[Union[dict, tuple]] = None, *, connection=None) -> list[dict[str, Any]]:
    owns = connection is None
    con = connection or connect()
    try:
        with con.cursor() as cur:
            cur.execute(sql, binds or {})
            names = [item[0] for item in cur.description or []]
            return [dict(zip(names, row)) for row in cur.fetchall()]
    finally:
        if owns:
            con.close()


def execute(sql: str, binds: Optional[Union[dict, tuple, list]] = None, *, many: bool = False, connection=None):
    owns = connection is None
    con = connection or connect()
    try:
        with con.cursor() as cur:
            if many:
                cur.executemany(sql, binds or [])
            else:
                cur.execute(sql, binds or {})
        con.commit()
    finally:
        if owns:
            con.close()


def ddl(sql: str, *, ignore: tuple[str, ...] = ("ORA-00955", "ORA-01430")) -> bool:
    try:
        execute(sql)
        return True
    except oracledb.DatabaseError as exc:
        if any(code in str(exc) for code in ignore):
            return False
        raise


with connect() as con:
    version_row = fetch_all(
        "SELECT product, version_full FROM product_component_version "
        "WHERE product LIKE 'Oracle%Database%' FETCH FIRST 1 ROW ONLY",
        connection=con,
    )[0]
print("Connected:", version_row)


ERPA_AGENT default tablespace: ERPA_DATA (automatic segment space management).
Connected: {'PRODUCT': 'Oracle AI Database 26ai Free', 'VERSION_FULL': '23.26.0.0.0'}


In [5]:
EMBED_ONNX_URL = os.getenv(
    "ERPA_EMBED_ONNX_URL",
    "https://objectstorage.us-ashburn-1.oraclecloud.com/n/adwc4pm/b/OML-Resources/o/all_MiniLM_L12_v2.onnx",
)
model_present = bool(fetch_all(
    "SELECT 1 FROM user_mining_models WHERE model_name=:name",
    {"name": CFG.embed_model.upper()},
))

if not model_present:
    model_dir = Path(tempfile.gettempdir()) / "erpa-oracle-models"
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / "all_MiniLM_L12_v2.onnx"
    if not model_path.exists() or model_path.stat().st_size < 1_000_000:
        with requests.get(EMBED_ONNX_URL, stream=True, timeout=600) as response:
            response.raise_for_status()
            with model_path.open("wb") as output:
                for chunk in response.iter_content(1 << 20):
                    output.write(chunk)
    model_bytes = model_path.read_bytes()
    metadata = {
        "function": "embedding",
        "embeddingOutput": "embedding",
        "input": {"input": ["DATA"]},
    }
    with connect() as con:
        lob = con.createlob(oracledb.DB_TYPE_BLOB)
        lob.write(model_bytes)
        with con.cursor() as cur:
            cur.execute(
                "BEGIN DBMS_VECTOR.LOAD_ONNX_MODEL(:name,:data,JSON(:metadata)); END;",
                {"name": CFG.embed_model, "data": lob, "metadata": json.dumps(metadata)},
            )
        con.commit()

embedding = fetch_all(
    f"SELECT VECTOR_EMBEDDING({CFG.embed_model} USING 'ERPA smoke test' AS DATA) AS v FROM dual"
)[0]["V"]
assert len(embedding) == CFG.embed_dim
print(f"{CFG.embed_model} is loaded and returns {len(embedding)} dimensions inside Oracle.")


ALL_MINILM_L12_V2 is loaded and returns 384 dimensions inside Oracle.


# Part 2 · Seed the ERPA business world

The fixture is compact but intentionally designed around decisions, not decorative sample data. Four rows are below reorder point, but Berlin ThermaCore M is already covered by `PO-BER-THC-OPEN`. A correct brief therefore produces three new actions and one suppression. WarmLayer has four stable UK observations followed by a sharp increase with no internal promotion. Margin facts deliberately rank UK first by gross margin even though EU has more units and revenue, testing whether the harness uses governed meaning instead of a plausible proxy.

| Data set | Decision it supports | Important integrity rule |
|---|---|---|
| Products + inventory | Stock position by SKU, region, city, and size | SKU is the grain; product lines are referenced |
| Purchase orders | Suppress already-covered replenishment issues | PO points to the exact inventory SKU |
| Weekly sales | Compare current movement to prior complete observations | Product, region, and week form the key |
| Margin facts | Rank regions under a governed metric | Region is the reporting grain |
| Institutional documents | Retrieve company guidance and review notes | Every result carries a source URL |
| External signals | Add dated outside evidence | Signal time, region, type, and URL are retained |
| Action audit | Record drafted, approved, and executed effects | One immutable action ID owns the transition |

The cell deletes and reloads only these fixture tables, making scenario assertions deterministic. It does not reset memory/checkpoint packages, which are separately namespaced and inspected later.

In [6]:
for statement in [
    """CREATE TABLE erpa_products (
        product_line VARCHAR2(40) PRIMARY KEY, category VARCHAR2(40) NOT NULL,
        unit_cost NUMBER(10,2) NOT NULL, unit_price NUMBER(10,2) NOT NULL)""",
    """CREATE TABLE erpa_inventory (
        sku VARCHAR2(80) PRIMARY KEY, product_line VARCHAR2(40) REFERENCES erpa_products,
        category VARCHAR2(40), region VARCHAR2(20), city VARCHAR2(40), size_code VARCHAR2(8),
        on_hand NUMBER, reorder_point NUMBER)""",
    """CREATE TABLE erpa_purchase_orders (
        po_id VARCHAR2(80) PRIMARY KEY, sku VARCHAR2(80) REFERENCES erpa_inventory,
        quantity NUMBER, status VARCHAR2(20), created_at DATE)""",
    """CREATE TABLE erpa_weekly_sales (
        product_line VARCHAR2(40), region VARCHAR2(20), week_start DATE, units NUMBER,
        promotion_flag CHAR(1), PRIMARY KEY(product_line,region,week_start))""",
    """CREATE TABLE erpa_margin_facts (
        region VARCHAR2(20) PRIMARY KEY, units NUMBER, net_revenue NUMBER(14,2),
        gross_margin NUMBER(14,2), avg_discount_percent NUMBER(5,2))""",
    """CREATE TABLE erpa_institutional_docs (
        doc_id VARCHAR2(80) PRIMARY KEY, title VARCHAR2(200), body CLOB,
        source_url VARCHAR2(500), embedding VECTOR(384,FLOAT32))""",
    """CREATE TABLE erpa_external_signals (
        signal_id VARCHAR2(80) PRIMARY KEY, observed_at DATE, region VARCHAR2(20),
        signal_type VARCHAR2(40), summary VARCHAR2(1000), source_url VARCHAR2(500))""",
    """CREATE TABLE erpa_action_audit (
        action_id VARCHAR2(80) PRIMARY KEY, action_type VARCHAR2(40), payload CLOB,
        status VARCHAR2(30), created_at TIMESTAMP DEFAULT SYSTIMESTAMP,
        approved_at TIMESTAMP, executed_at TIMESTAMP)""",
]:
    ddl(statement)

products = [
    ("ThermaCore", "Outerwear", 41.00, 129.90),
    ("FieldShell", "Outerwear", 32.00, 99.90),
    ("PleatCraft", "Tops", 18.00, 69.90),
    ("WarmLayer", "Innerwear", 4.80, 19.90),
    ("CarryAll", "Accessories", 6.20, 24.90),
]
inventory = [
    ("THC-UK-XS", "ThermaCore", "Outerwear", "UK", "London", "XS", 100, 20),
    ("THC-UK-M", "ThermaCore", "Outerwear", "UK", "London", "M", 35, 30),
    ("THC-UK-L", "ThermaCore", "Outerwear", "UK", "London", "L", 5, 30),
    ("THC-UK-XXL", "ThermaCore", "Outerwear", "UK", "London", "XXL", 90, 20),
    ("THC-EU-PAR-XS", "ThermaCore", "Outerwear", "EU", "Paris", "XS", 70, 20),
    ("THC-EU-PAR-M", "ThermaCore", "Outerwear", "EU", "Paris", "M", 35, 30),
    ("THC-EU-PAR-L", "ThermaCore", "Outerwear", "EU", "Paris", "L", 31, 30),
    ("THC-EU-PAR-XXL", "ThermaCore", "Outerwear", "EU", "Paris", "XXL", 80, 20),
    ("THC-EU-BER-M", "ThermaCore", "Outerwear", "EU", "Berlin", "M", 4, 30),
    ("FSH-UK-M", "FieldShell", "Outerwear", "UK", "London", "M", 6, 32),
    ("PCT-EU-M", "PleatCraft", "Tops", "EU", "Paris", "M", 8, 35),
    ("CAL-UK-M", "CarryAll", "Accessories", "UK", "London", "M", 2, 20),
]
today = date.today()
this_monday = today - timedelta(days=today.weekday())
weekly = [
    ("WarmLayer", "UK", this_monday - timedelta(weeks=4), 100, "N"),
    ("WarmLayer", "UK", this_monday - timedelta(weeks=3), 105, "N"),
    ("WarmLayer", "UK", this_monday - timedelta(weeks=2), 95, "N"),
    ("WarmLayer", "UK", this_monday - timedelta(weeks=1), 100, "N"),
    ("WarmLayer", "UK", this_monday, 320, "N"),
]
margins = [
    ("UK", 1200, 125000, 48000, 5.0),
    ("US", 1050, 118000, 41000, 7.0),
    ("EU", 1400, 132000, 36000, 12.0),
]
docs = [
    ("regional-merch", "Regional merchandising notes",
     "UK customers respond quickly to temperature changes. Present WarmLayer when mean temperatures fall below 12°C. Report UK values in GBP.",
     "oracle://erpa/institutional/regional-merch"),
    ("restock-playbook", "Restock playbook",
     "Do not re-raise an inventory issue when an open purchase order already covers it. The morning brief must cite the purchase order.",
     "oracle://erpa/institutional/restock-playbook"),
    ("size-curve", "Size curve guidance",
     "ThermaCore M and L are the commercial centre. XS and XXL overhang must not hide a core-size sell-out.",
     "oracle://erpa/institutional/size-curve"),
]
signals = [
    ("uk-cold-snap", today, "UK", "weather",
     "A UK cold snap pushed mean temperatures below the 12°C presentation threshold.",
     "https://example.test/weather/uk-cold-snap"),
]

with connect() as con:
    cur = con.cursor()
    for table in ("erpa_action_audit", "erpa_external_signals", "erpa_institutional_docs",
                  "erpa_margin_facts", "erpa_weekly_sales", "erpa_purchase_orders",
                  "erpa_inventory", "erpa_products"):
        cur.execute(f"DELETE FROM {table}")
    cur.executemany("INSERT INTO erpa_products VALUES (:1,:2,:3,:4)", products)
    cur.executemany("INSERT INTO erpa_inventory VALUES (:1,:2,:3,:4,:5,:6,:7,:8)", inventory)
    cur.execute("INSERT INTO erpa_purchase_orders VALUES (:1,:2,:3,:4,:5)",
                ("PO-BER-THC-OPEN", "THC-EU-BER-M", 600, "OPEN", today - timedelta(days=1)))
    cur.executemany("INSERT INTO erpa_weekly_sales VALUES (:1,:2,:3,:4,:5)", weekly)
    cur.executemany("INSERT INTO erpa_margin_facts VALUES (:1,:2,:3,:4,:5)", margins)
    document_binds = [
        {"doc_id": doc_id, "title": title, "body": body, "source_url": source_url}
        for doc_id, title, body, source_url in docs
    ]
    cur.executemany(
        f"""INSERT INTO erpa_institutional_docs(doc_id,title,body,source_url,embedding)
        VALUES (:doc_id,:title,:body,:source_url,
          VECTOR_EMBEDDING({CFG.embed_model} USING :body AS DATA))""",
        document_binds,
    )
    cur.executemany("INSERT INTO erpa_external_signals VALUES (:1,:2,:3,:4,:5,:6)", signals)
    con.commit()

pd.DataFrame(fetch_all("SELECT * FROM erpa_inventory ORDER BY product_line,region,city,size_code"))


,SKU,PRODUCT_LINE,CATEGORY,REGION,CITY,SIZE_CODE,ON_HAND,REORDER_POINT
0,CAL-UK-M,CarryAll,Accessories,UK,London,M,2,20
1,FSH-UK-M,FieldShell,Outerwear,UK,London,M,6,32
2,PCT-EU-M,PleatCraft,Tops,EU,Paris,M,8,35
3,THC-EU-BER-M,ThermaCore,Outerwear,EU,Berlin,M,4,30
4,THC-EU-PAR-L,ThermaCore,Outerwear,EU,Paris,L,31,30
5,THC-EU-PAR-M,ThermaCore,Outerwear,EU,Paris,M,35,30
6,THC-EU-PAR-XS,ThermaCore,Outerwear,EU,Paris,XS,70,20
7,THC-EU-PAR-XXL,ThermaCore,Outerwear,EU,Paris,XXL,80,20
8,THC-UK-L,ThermaCore,Outerwear,UK,London,L,5,30
9,THC-UK-M,ThermaCore,Outerwear,UK,London,M,35,30


# Part 3 · Build a living semantic layer

> **Definition — semantic layer**
>
> A semantic layer is a governed translation between business language and physical data. It defines canonical metrics, grains, joins, filters, synonyms, and evidence rules so “most profitable,” “covered stock,” or “latest week” has one inspectable meaning. In an agent harness it is retrieved context and executable authority: it helps the model choose the correct object, while trusted views and tools enforce the definition at query time.

Without this layer, a capable model might rank regions by revenue, units, margin percentage, or gross margin—all syntactically valid but operationally different. ERPA uses four sources of meaning:

1. **Governed views** encode formulas and safe joins once.
2. **Oracle table and column comments** keep definitions beside the schema they describe.
3. **Curated semantic hints** capture synonyms, metric policy, suppression logic, and approved query shapes that DDL cannot express cleanly.
4. **`V$SQL` workload evidence** shows how the application is actually querying ERPA objects, allowing the catalog to evolve with real use.

Those facts are embedded into an Oracle vector catalog. At run time, only the closest semantic entries are assembled into the prompt; the model does not receive an unbounded schema dump. A database procedure and Scheduler job periodically refresh schema/hint entries.

```mermaid
flowchart TB
    Q[User business question] --> AS[LangGraph assemble_context]
    subgraph Build[Living semantic layer build path]
      V[Governed views and formulas] --> SCAN[Catalog scanner]
      C[Table and column comments] --> SCAN
      H[Curated hints and synonyms] --> SCAN
      W[V$SQL workload shapes] --> SCAN
      SCAN --> EMB[Oracle VECTOR_EMBEDDING]
      EMB --> CAT[(ERPA_SEMANTIC_CATALOG)]
      JOB[Oracle Scheduler refresh] --> SCAN
    end
    Q --> SEARCH[Cosine vector search]
    CAT --> SEARCH
    SEARCH --> AS
    AS --> LLM[Claude chooses a trusted tool]
    LLM --> TOOL[Allowlisted SQL tool]
    TOOL --> VIEW[Governed Oracle view/table]
    VIEW --> EVIDENCE[Grounded rows]
    EVIDENCE --> LLM
    LLM --> A[Business answer]
```

Notice the two enforcement points. Vector retrieval teaches the model which meaning is relevant; the governed view/tool computes that meaning. Prompt text alone is not a sufficient control for a financial metric.

## Encode canonical business meaning in Oracle

The first semantic object is `ERPA_V_PROFITABILITY`. It exposes the source measures and computes `gross_margin_percent` with a zero-safe denominator. ERPA's canonical ranking still uses the `gross_margin` amount; the percentage is available for a different question rather than silently substituting for it.

Oracle comments make the schema self-describing to humans, database tools, and the catalog scanner. The comment on `gross_margin` resolves the phrase “most profitable,” while the `promotion_flag` comment prevents a common causal error: only `Y` proves that an internal promotion was active. The view and comments are executable metadata, survive notebook sessions, and can be inspected independently of the model prompt.

This pattern separates three concerns: source tables retain facts, a view presents governed calculations, and comments state meaning. Changing a definition becomes a database governance change rather than an improvised prompt edit.

In [7]:
ddl("""CREATE OR REPLACE VIEW erpa_v_profitability AS
    SELECT region, units, net_revenue, gross_margin, avg_discount_percent,
           ROUND(gross_margin / NULLIF(net_revenue,0) * 100, 2) AS gross_margin_percent
    FROM erpa_margin_facts""", ignore=())
execute("COMMENT ON TABLE erpa_margin_facts IS 'Governed quarterly regional commercial facts'")
execute("COMMENT ON COLUMN erpa_margin_facts.gross_margin IS 'Net revenue minus product cost; canonical meaning of profitable'")
execute("COMMENT ON COLUMN erpa_weekly_sales.promotion_flag IS 'Y only when an internal promotion was active that week'")


## Create the curated and generated catalog stores

`ERPA_SEMANTIC_HINTS` stores small, human-reviewed rules. `subject` is a stable identifier such as `metric:profitability`; `hint_type` supports inspection by class; `priority` leaves room for future ranking policy; and `hint_text` is the retrievable explanation.

`ERPA_SEMANTIC_CATALOG` is the generated retrieval index. Each row records a stable catalog ID, its source type, readable text, refresh timestamp, and a 384-dimensional `FLOAT32` vector compatible with the ONNX model loaded in Part 1. Keeping readable text beside the vector is essential: a vector is a search mechanism, not evidence a reviewer can understand.

The two-table design prevents regeneration from erasing governance. Curators edit the hints table; the scanner rebuilds the catalog. In a production design, ownership and approval permissions for hints would be stricter than permissions for the refresh procedure.

In [8]:

ddl("""CREATE TABLE erpa_semantic_hints (
    subject VARCHAR2(200) PRIMARY KEY, hint_type VARCHAR2(40), hint_text VARCHAR2(2000),
    priority NUMBER DEFAULT 100)""")
ddl("""CREATE TABLE erpa_semantic_catalog (
    catalog_id VARCHAR2(200) PRIMARY KEY, source_type VARCHAR2(40), catalog_text VARCHAR2(4000),
    refreshed_at TIMESTAMP DEFAULT SYSTIMESTAMP, embedding VECTOR(384,FLOAT32))""")


False

## Seed policy hints and a discoverable workload shape

The curated hints cover four high-value ambiguities:

- **metric:** “profitability” means gross-margin amount, not revenue;
- **join:** low inventory must be reconciled to open POs by SKU;
- **scope:** the recurring brief has an explicit region/category boundary;
- **query pattern:** the WarmLayer spike compares the latest row to preceding complete observations and checks the promotion flag.

The cell replaces the hint set transactionally, then executes a harmless governed profitability query tagged with an `ERPA_METRIC` SQL comment. Oracle records the statement in `V$SQL`, giving the later scanner a meaningful workload example. This is a lightweight form of a living layer: the catalog reflects both declared policy and observed use.

Workload SQL is not automatically trusted policy. It is labeled as `workload` evidence and retrieved alongside, rather than promoted over, curated hints and governed views.

In [9]:

hints = [
    ("metric:profitability", "metric", "Most profitable means highest gross_margin, never highest revenue.", 1000),
    ("join:inventory_po", "join", "Join ERPA_INVENTORY.SKU to ERPA_PURCHASE_ORDERS.SKU and suppress OPEN purchase orders.", 900),
    ("scope:morning_brief", "filter", "Morning brief scope is UK and EU, recurring categories Outerwear and Tops, excluding remembered categories.", 800),
    ("query:warm_spike", "query", "Compare the latest WarmLayer UK week with the preceding four complete weekly observations and inspect promotion_flag.", 700),
]
with connect() as con:
    cur = con.cursor()
    cur.execute("DELETE FROM erpa_semantic_hints")
    cur.executemany("INSERT INTO erpa_semantic_hints VALUES (:1,:2,:3,:4)", hints)
    # Seed a meaningful workload statement whose SQL comment can later be discovered through V$SQL.
    cur.execute("SELECT /* ERPA_METRIC gross_margin profitability */ region,gross_margin FROM erpa_v_profitability ORDER BY gross_margin DESC")
    cur.fetchall()
    con.commit()


## Scan metadata, hints, and workload into one vector space

`refresh_semantic_catalog()` performs an extract–normalize–embed–load cycle:

1. read every ERPA column, data type, and optional Oracle comment;
2. append curated hint rows;
3. read the most frequently executed ERPA-shaped statements from `V$SQL` when the grant is available;
4. deduplicate by stable `catalog_id` because one SQL ID can expose multiple child cursors;
5. replace the generated catalog in one transaction; and
6. compute every embedding inside the `INSERT` with `VECTOR_EMBEDDING`.

The `V$SQL` scan is intentionally non-fatal. Missing workload privileges should reduce the living signal, not prevent the governed schema and hints from working. Other database failures remain visible.

For a larger catalog, use staging plus `MERGE` rather than delete/reload, record source hashes, and refresh only changed entries. The compact workshop version makes the full data lineage easier to inspect.

In [10]:
def refresh_semantic_catalog() -> int:
    facts: list[tuple[str, str, str]] = []
    for row in fetch_all("""SELECT c.table_name,c.column_name,c.data_type,cc.comments
        FROM user_tab_columns c
        LEFT JOIN user_col_comments cc ON cc.table_name=c.table_name AND cc.column_name=c.column_name
        WHERE c.table_name LIKE 'ERPA_%'"""):
        text = f"{row['TABLE_NAME']}.{row['COLUMN_NAME']} ({row['DATA_TYPE']})"
        if row["COMMENTS"]:
            text += f" -- {row['COMMENTS']}"
        facts.append((f"column:{row['TABLE_NAME']}.{row['COLUMN_NAME']}", "column", text))
    for row in fetch_all("SELECT subject,hint_type,hint_text FROM erpa_semantic_hints"):
        facts.append((row["SUBJECT"], "hint", f"{row['HINT_TYPE']}: {row['HINT_TEXT']}"))
    try:
        workload = fetch_all("""SELECT sql_id,sql_text,executions FROM v$sql
            WHERE parsing_schema_name=USER AND UPPER(sql_text) LIKE '%ERPA_%'
            ORDER BY executions DESC FETCH FIRST 10 ROWS ONLY""")
        for row in workload:
            facts.append((f"vsql:{row['SQL_ID']}", "workload",
                          f"Executed {row['EXECUTIONS']} times: {str(row['SQL_TEXT'])[:700]}"))
    except oracledb.DatabaseError as exc:
        print("V$SQL scan warning:", str(exc).splitlines()[0])
    with connect() as con:
        cur = con.cursor()
        cur.execute("DELETE FROM erpa_semantic_catalog")
        # V$SQL may expose multiple child cursors for one SQL_ID. The catalog
        # keeps one current semantic observation per stable catalog identifier.
        unique_facts = {
            catalog_id: (source_type, catalog_text)
            for catalog_id, source_type, catalog_text in facts
        }
        fact_binds = [
            {"catalog_id": catalog_id, "source_type": value[0], "catalog_text": value[1]}
            for catalog_id, value in unique_facts.items()
        ]
        cur.executemany(
            f"""INSERT INTO erpa_semantic_catalog(catalog_id,source_type,catalog_text,embedding)
                VALUES (:catalog_id,:source_type,:catalog_text,
                  VECTOR_EMBEDDING({CFG.embed_model} USING :catalog_text AS DATA))""",
            fact_binds,
        )
        con.commit()
    return len(fact_binds)


## Retrieve meaning before selecting tools

`semantic_search` embeds the user's phrase with the same Oracle model and orders catalog rows with cosine `VECTOR_DISTANCE`. `FETCH APPROX FIRST` lets Oracle use an available vector index while preserving the requested top-k contract.

The demonstration asks which metric means “most profitable.” Expected nearest evidence includes the curated profitability hint and the commented gross-margin column. That result is not itself the financial answer; it tells the graph which governed meaning and object should shape the next tool call.

Distance is interpreted comparatively here: lower is more similar. Production retrieval should also enforce tenant/source filters, minimum evidence classes, evaluation-derived cutoffs, and citations back to the originating DDL or governance record.

In [11]:

def semantic_search(query: str, k: int = 6) -> list[dict[str, Any]]:
    return fetch_all(f"""SELECT catalog_id,source_type,catalog_text,
        VECTOR_DISTANCE(embedding,VECTOR_EMBEDDING({CFG.embed_model} USING :query AS DATA),COSINE) AS distance
        FROM erpa_semantic_catalog ORDER BY distance FETCH APPROX FIRST :k ROWS ONLY""",
        {"query": query, "k": k})


print("Semantic catalog facts:", refresh_semantic_catalog())
pd.DataFrame(semantic_search("which governed metric means most profitable", 4))


Semantic catalog facts: 150


,CATALOG_ID,SOURCE_TYPE,CATALOG_TEXT,DISTANCE
0,metric:profitability,hint,metric: Most profitable means highest gross_ma...,0.281643
1,column:ERPA_MARGIN_FACTS.GROSS_MARGIN,column,ERPA_MARGIN_FACTS.GROSS_MARGIN (NUMBER) -- Net...,0.449428
2,column:ERPA_V_PROFITABILITY.GROSS_MARGIN,column,ERPA_V_PROFITABILITY.GROSS_MARGIN (NUMBER),0.529316
3,column:ERPA_V_PROFITABILITY.GROSS_MARGIN_PERCENT,column,ERPA_V_PROFITABILITY.GROSS_MARGIN_PERCENT (NUM...,0.539196


## Keep the semantic layer alive with Oracle Scheduler

The Python scanner demonstrates all four sources, including `V$SQL`. The stored procedure implements the database-native refresh path for the stable schema and hint sources, which can run without a notebook process. It deletes only generated `column` and `hint` rows, then recreates their text and vectors in Oracle.

`create_or_replace_job` makes Scheduler setup rerunnable: it drops only the named job, recreates it as a PL/SQL block, and enables a six-hour calendar. The resulting `USER_SCHEDULER_JOBS` rows are displayed so schedule state is evidence rather than an assumption.

The procedure deliberately leaves workload harvesting to the richer Python scanner. A production implementation could add a reviewed PL/SQL workload stage, an incremental change table, or a job chain. Separating the two paths makes their trust and privilege requirements explicit.

In [12]:
ddl(f"""CREATE OR REPLACE PROCEDURE erpa_refresh_semantic_catalog AS
BEGIN
  DELETE FROM erpa_semantic_catalog WHERE source_type IN ('column','hint');
  INSERT INTO erpa_semantic_catalog(catalog_id,source_type,catalog_text,embedding)
  SELECT 'column:'||c.table_name||'.'||c.column_name, 'column',
         c.table_name||'.'||c.column_name||' ('||c.data_type||') '||NVL('-- '||cc.comments,''),
         VECTOR_EMBEDDING({CFG.embed_model} USING
           c.table_name||'.'||c.column_name||' '||NVL(cc.comments,'') AS DATA)
  FROM user_tab_columns c
  LEFT JOIN user_col_comments cc ON cc.table_name=c.table_name AND cc.column_name=c.column_name
  WHERE c.table_name LIKE 'ERPA_%';
  INSERT INTO erpa_semantic_catalog(catalog_id,source_type,catalog_text,embedding)
  SELECT subject,'hint',hint_type||': '||hint_text,
         VECTOR_EMBEDDING({CFG.embed_model} USING hint_type||': '||hint_text AS DATA)
  FROM erpa_semantic_hints;
  COMMIT;
END;""", ignore=())


def create_or_replace_job(name: str, action: str, repeat_interval: str) -> None:
    try:
        execute("BEGIN DBMS_SCHEDULER.DROP_JOB(:name,force=>TRUE); END;", {"name": name})
    except oracledb.DatabaseError:
        pass
    execute("""BEGIN DBMS_SCHEDULER.CREATE_JOB(
        job_name=>:name, job_type=>'PLSQL_BLOCK', job_action=>:action,
        repeat_interval=>:repeat_interval, enabled=>TRUE); END;""",
        {"name": name, "action": action, "repeat_interval": repeat_interval})


create_or_replace_job(
    "ERPA_SEMANTIC_REFRESH_JOB", "BEGIN erpa_refresh_semantic_catalog; END;",
    "FREQ=HOURLY;INTERVAL=6",
)
jobs = fetch_all("SELECT job_name,enabled,repeat_interval FROM user_scheduler_jobs WHERE job_name LIKE 'ERPA_%'")
pd.DataFrame(jobs)


,JOB_NAME,ENABLED,REPEAT_INTERVAL
0,ERPACUSTOM_PURGE_EXPIRED_RECORDS_J,TRUE,FREQ=DAILY;INTERVAL=1
1,ERPA_SEMANTIC_REFRESH_JOB,TRUE,FREQ=HOURLY;INTERVAL=6


# Part 4 · SecureFile ScratchFS: short-term working memory

Plans, intermediate results, drafts, and large tool artifacts are useful during a run but do not automatically deserve durable semantic recall. `ScratchFS` gives ERPA a small POSIX-like interface backed by Oracle SecureFile BLOB rows. The data is transactional, scoped by logical session, observable from any database connection, and retained with the rest of the workshop evidence.

| State | Typical content | Lifetime | Retrieval style |
|---|---|---|---|
| Scratch file | current plan, draft, raw E2B output | one logical session unless retained | exact path, list, tail, grep |
| Graph checkpoint | node state and pending interrupt | one checkpoint thread | exact `thread_id` |
| OAMP memory | accepted preference, fact, guideline | cross-session with retention policy | hybrid semantic/lexical search |
| Business row | inventory, PO, margin, audit fact | system-of-record lifecycle | governed SQL |

Treating scratch as a distinct tier reduces prompt growth and memory pollution. Promotion is explicit: only files marked `promote_on_end='Y'` are chunked and staged when the session transitions from `ACTIVE` to `ENDED`.

The three backing tables represent the lifecycle. `ERPA_AGENT_SESSIONS` owns the logical boundary, `ERPA_SCRATCH_FILES` stores directories and SecureFile content, and `ERPA_MEMORY_PROMOTION_QUEUE` decouples an Oracle trigger from the Python OAMP client.

In [13]:
ddl("""CREATE TABLE erpa_agent_sessions (
    session_id VARCHAR2(80) PRIMARY KEY, user_id VARCHAR2(80), thread_id VARCHAR2(120),
    status VARCHAR2(20) DEFAULT 'ACTIVE', started_at TIMESTAMP DEFAULT SYSTIMESTAMP,
    ended_at TIMESTAMP)""")
ddl("""CREATE TABLE erpa_scratch_files (
    session_id VARCHAR2(80) REFERENCES erpa_agent_sessions, path VARCHAR2(400),
    content BLOB, is_dir CHAR(1) DEFAULT 'N', promotion_state CHAR(1) DEFAULT 'N',
    promote_on_end CHAR(1) DEFAULT 'N', updated_at TIMESTAMP DEFAULT SYSTIMESTAMP,
    PRIMARY KEY(session_id,path)) LOB(content) STORE AS SECUREFILE""")
ddl("""CREATE TABLE erpa_memory_promotion_queue (
    queue_id RAW(16) DEFAULT SYS_GUID() PRIMARY KEY, session_id VARCHAR2(80),
    path VARCHAR2(400), chunk CLOB, consumed CHAR(1) DEFAULT 'N',
    staged_at TIMESTAMP DEFAULT SYSTIMESTAMP, consumed_at TIMESTAMP)""")


False

## Implement a safe file-like adapter over SecureFile rows

`ScratchFS` exposes only operations the harness needs: `mkdir`, `write`, `read`, `append`, `exists`, `list`, `tail`, and `grep`. It is not a host filesystem mount and cannot access notebook files.

The path normalizer adds the `/scratch` mount, uses POSIX semantics on every platform, and rejects traversal components. Each lookup includes `session_id`, so identical paths in two sessions do not collide. Writes use Oracle `MERGE`, which makes plan updates idempotent and resets promotion state when content changes. Text is encoded as UTF-8 into a BLOB and decoded predictably on read.

`list`, `tail`, and `grep` deliberately execute in trusted Python over rows already scoped by session. User text is passed as a regex value to Python—not concatenated into SQL. For large production files, grep/tail should use streaming LOB reads or Oracle Text rather than loading every file.

The following cells create a session and its standard directories, write a current plan, and mark one supplier note for promotion. A fresh `ScratchFS` instance reads the note, proving that state is in Oracle rather than an instance attribute.

In [14]:


class ScratchFS:
    """A session-scoped POSIX-like interface over Oracle SecureFile rows."""
    def __init__(self, session_id: str, mount: str = "/scratch"):
        self.session_id = session_id
        self.mount = mount.rstrip("/")

    def _abs(self, path: str) -> str:
        clean = "/" + str(PurePosixPath("/" + path.strip("/"))).lstrip("/")
        absolute = clean if clean.startswith(self.mount + "/") else self.mount + clean
        if ".." in PurePosixPath(absolute).parts:
            raise ValueError("Scratch paths cannot escape their mount")
        return absolute

    def mkdir(self, path: str) -> None:
        execute("""MERGE INTO erpa_scratch_files d
          USING (SELECT :session_id session_id,:path path FROM dual) s
          ON (d.session_id=s.session_id AND d.path=s.path)
          WHEN NOT MATCHED THEN INSERT(session_id,path,is_dir) VALUES(s.session_id,s.path,'Y')""",
          {"session_id": self.session_id, "path": self._abs(path)})

    def write(self, path: str, content: str, *, promote_on_end: bool = False) -> dict[str, Any]:
        data = content.encode("utf-8")
        execute("""MERGE INTO erpa_scratch_files d
          USING (SELECT :session_id session_id,:path path FROM dual) s
          ON (d.session_id=s.session_id AND d.path=s.path)
          WHEN MATCHED THEN UPDATE SET content=:content,is_dir='N',promotion_state='N',
            promote_on_end=:promote,updated_at=SYSTIMESTAMP
          WHEN NOT MATCHED THEN INSERT(session_id,path,content,is_dir,promote_on_end)
            VALUES(s.session_id,s.path,:content,'N',:promote)""",
          {"session_id": self.session_id, "path": self._abs(path), "content": data,
           "promote": "Y" if promote_on_end else "N"})
        return {"path": self._abs(path), "bytes": len(data), "promote_on_end": promote_on_end}

    def read(self, path: str) -> str:
        rows = fetch_all("SELECT content FROM erpa_scratch_files WHERE session_id=:s AND path=:p",
                         {"s": self.session_id, "p": self._abs(path)})
        if not rows:
            raise FileNotFoundError(self._abs(path))
        value = rows[0]["CONTENT"] or b""
        return value.decode("utf-8", errors="replace") if isinstance(value, bytes) else str(value)

    def append(self, path: str, content: str, *, promote_on_end: bool = False) -> dict[str, Any]:
        prior = self.read(path) if self.exists(path) else ""
        return self.write(path, prior + content, promote_on_end=promote_on_end)

    def exists(self, path: str) -> bool:
        return bool(fetch_all("SELECT 1 FROM erpa_scratch_files WHERE session_id=:s AND path=:p",
                              {"s": self.session_id, "p": self._abs(path)}))

    def list(self, root: str = "/") -> list[str]:
        prefix = self._abs(root).rstrip("/") + "/%"
        return [r["PATH"] for r in fetch_all(
            "SELECT path FROM erpa_scratch_files WHERE session_id=:s AND path LIKE :p AND is_dir='N' ORDER BY path",
            {"s": self.session_id, "p": prefix})]

    def tail(self, path: str, n: int = 20) -> str:
        return "\n".join(self.read(path).splitlines()[-n:])

    def grep(self, pattern: str, root: str = "/") -> List[Dict[str, Any]]:
        rx = re.compile(pattern, re.IGNORECASE)
        hits = []
        for path in self.list(root):
            for number, line in enumerate(self.read(path).splitlines(), 1):
                if rx.search(line):
                    hits.append({"path": path, "line": number, "text": line[:300]})
        return hits


In [15]:
def begin_session(session_id: str, thread_id: str, user_id: str = CFG.user_id) -> ScratchFS:
    execute("""MERGE INTO erpa_agent_sessions d
      USING (SELECT :session_id session_id FROM dual) s ON(d.session_id=s.session_id)
      WHEN MATCHED THEN UPDATE SET user_id=:user_id,thread_id=:thread_id,status='ACTIVE',ended_at=NULL
      WHEN NOT MATCHED THEN INSERT(session_id,user_id,thread_id) VALUES(:session_id,:user_id,:thread_id)""",
      {"session_id": session_id, "user_id": user_id, "thread_id": thread_id})
    fs = ScratchFS(session_id)
    for directory in ("/plans", "/notes", "/tool_out", "/inbox"):
        fs.mkdir(directory)
    return fs


In [16]:
SCRATCH_DEMO_SESSION = "scratch-demo"
fs_demo = begin_session(SCRATCH_DEMO_SESSION, "scratch-demo-thread")
fs_demo.write("/plans/current.md", "# Plan\n1. Inspect stock.\n2. Reconcile open purchase orders.\n")
SCRATCH_PROMOTION_TOKEN = f"promotion-{uuid.uuid4().hex}"
fs_demo.write(
    "/notes/supplier.md",
    f"Berlin ThermaCore is covered by PO-BER-THC-OPEN. Verification token: {SCRATCH_PROMOTION_TOKEN}.",
    promote_on_end=True,
)
assert "PO-BER" in ScratchFS(SCRATCH_DEMO_SESSION).read("/notes/supplier.md")
print(fs_demo.list("/"), fs_demo.grep("purchase|PO-BER", "/notes"))


['/scratch/notes/supplier.md', '/scratch/plans/current.md'] [{'path': '/scratch/notes/supplier.md', 'line': 1, 'text': 'Berlin ThermaCore is covered by PO-BER-THC-OPEN. Verification token: promotion-91b4052fc7a04c02ba73fd39d95b0a01.'}]


## Promote selected scratch files when a logical session ends

A database connection ending is not the same event as an agent session ending—connection pools open and close connections throughout one conversation. The application therefore owns an explicit `ACTIVE → ENDED` transition in `ERPA_AGENT_SESSIONS`.

The promotion flow has two phases:

```mermaid
sequenceDiagram
    participant App as Harness
    participant DB as Oracle session row
    participant Trigger as Session-end trigger
    participant Queue as Promotion queue
    participant Worker as OAMP consumer
    participant Memory as Oracle Agent Memory
    App->>DB: UPDATE status = ENDED
    DB->>Trigger: AFTER UPDATE OF status
    Trigger->>Queue: chunk eligible SecureFile content
    Note over Trigger,Queue: same Oracle transaction
    Worker->>Queue: read unconsumed chunks
    Worker->>Memory: add durable fact with provenance
    Worker->>Queue: mark consumed
    Worker->>DB: mark file promoted
```

ScratchFS writes are known UTF-8 text, so the trigger converts each SecureFile BLOB to a temporary CLOB with `DBMS_LOB.CONVERTTOCLOB`, then calls `DBMS_VECTOR_CHAIN.UTL_TO_CHUNKS` to produce bounded, overlapping chunks. This deliberately avoids `UTL_TO_TEXT`: that routine is designed to extract text from mixed binary document formats and requires Oracle Text's `BLOB_POLICY`, which is not installed in the `latest-lite` image. The trigger stages only files explicitly marked for promotion and changes their state from `N` to `S`. Python then writes to OAMP and changes the queue/file states to `Y`.

This split is necessary because a PL/SQL trigger cannot safely invoke the Python OAMP client or a remote model. The queue creates an auditable handoff and a natural retry point. Production consumers should add claim/lease state and idempotency keys; the notebook uses the queue RAW ID as the durable OAMP memory ID.

In [17]:
ddl("""CREATE OR REPLACE PROCEDURE erpa_stage_session_memory(p_session_id VARCHAR2) AS
  l_document CLOB;
  l_dest_offset PLS_INTEGER;
  l_src_offset PLS_INTEGER;
  l_lang_context PLS_INTEGER;
  l_warning PLS_INTEGER;
BEGIN
  FOR f IN (SELECT path,content FROM erpa_scratch_files
            WHERE session_id=p_session_id AND is_dir='N'
              AND promote_on_end='Y' AND promotion_state='N') LOOP
    DBMS_LOB.CREATETEMPORARY(l_document,TRUE,DBMS_LOB.CALL);
    l_dest_offset := 1;
    l_src_offset := 1;
    l_lang_context := DBMS_LOB.DEFAULT_LANG_CTX;
    DBMS_LOB.CONVERTTOCLOB(
      dest_lob => l_document, src_blob => f.content, amount => DBMS_LOB.LOBMAXSIZE,
      dest_offset => l_dest_offset, src_offset => l_src_offset,
      blob_csid => NLS_CHARSET_ID('AL32UTF8'), lang_context => l_lang_context,
      warning => l_warning
    );
    IF l_warning <> 0 THEN
      RAISE_APPLICATION_ERROR(-20001,'Scratch BLOB contains invalid UTF-8');
    END IF;
    INSERT INTO erpa_memory_promotion_queue(session_id,path,chunk)
    SELECT p_session_id,f.path,ch.chunk
    FROM TABLE(DBMS_VECTOR_CHAIN.UTL_TO_CHUNKS(
           l_document,
           JSON('{"by":"words","max":200,"overlap":20,"split":"recursively","normalize":"all"}')
         )) chunks,
         JSON_TABLE(chunks.column_value,'$'
           COLUMNS(chunk VARCHAR2(4000) PATH '$.chunk_data')) ch;
    UPDATE erpa_scratch_files SET promotion_state='S'
      WHERE session_id=p_session_id AND path=f.path;
    DBMS_LOB.FREETEMPORARY(l_document);
  END LOOP;
EXCEPTION
  WHEN OTHERS THEN
    IF DBMS_LOB.ISTEMPORARY(l_document)=1 THEN
      DBMS_LOB.FREETEMPORARY(l_document);
    END IF;
    RAISE;
END;""", ignore=())

ddl("""CREATE OR REPLACE TRIGGER erpa_session_end_promote
AFTER UPDATE OF status ON erpa_agent_sessions
FOR EACH ROW
WHEN (NEW.status = 'ENDED' AND OLD.status <> 'ENDED')
BEGIN
  erpa_stage_session_memory(:NEW.session_id);
END;""", ignore=())


True

In [18]:

def stage_session_end(session_id: str) -> int:
    execute("""UPDATE erpa_agent_sessions SET status='ENDED',ended_at=SYSTIMESTAMP
               WHERE session_id=:session_id AND status<>'ENDED'""", {"session_id": session_id})
    return fetch_all("SELECT COUNT(*) n FROM erpa_memory_promotion_queue WHERE session_id=:s AND consumed='N'",
                     {"s": session_id})[0]["N"]


staged = stage_session_end(SCRATCH_DEMO_SESSION)
assert staged >= 1
print({"session": SCRATCH_DEMO_SESSION, "staged_chunks": staged,
       "trigger": "ERPA_SESSION_END_PROMOTE"})


{'session': 'scratch-demo', 'staged_chunks': 2, 'trigger': 'ERPA_SESSION_END_PROMOTE'}


# Part 5 · Oracle Agent Memory

Oracle Agent Memory (OAMP) owns the assistant's cross-session memory substrate: episodic thread messages, durable facts/preferences/guidelines, background extraction, summaries, context cards, retention, and hybrid retrieval. It uses the same in-database embedding model as the semantic layer, but its records answer a different question: *what should this agent remember about the user and prior work?*

OAMP does not replace source-of-truth tables or checkpoints. Inventory stays in `ERPA_INVENTORY`; the graph's node state stays in OracleSaver; working notes stay in ScratchFS until explicitly promoted. OAMP receives only durable, useful statements with user, agent, tenant, source, and optional session provenance.

The notebook verifies the released package version before creating a connection pool and `OracleDBEmbedder`. Normalization supports consistent cosine/hybrid retrieval, and the pool lets background memory work use independent Oracle connections.

In [19]:
from oracleagentmemory.core import (
    BackgroundExtractionQueueFullBehavior,
    MemoryExtractionConfig,
    MemoryExtractionMode,
    MemoryRetentionConfig,
    OracleAgentMemory,
    OracleDBMemoryStore,
    SchemaPolicy,
    SearchIndexSyncMode,
    SearchStrategy,
)
from oracleagentmemory.core.embedders import OracleDBEmbedder
from oracleagentmemory.core.llms import Llm
from importlib.metadata import version as package_version
from packaging.version import Version

OAMP_VERSION = package_version("oracleagentmemory")
if Version(OAMP_VERSION) < Version("26.6.0"):
    raise RuntimeError(
        f"This notebook requires oracleagentmemory>=26.6.0; found {OAMP_VERSION}. "
        "Re-run the installation cell and restart the kernel."
    )

pool = oracledb.create_pool(
    user=CFG.oracle_user, password=CFG.oracle_password, dsn=CFG.oracle_dsn,
    min=1, max=8, increment=1,
)
oamp_embedder = OracleDBEmbedder(
    connection=pool, model=CFG.embed_model, input_name="DATA",
    embedding_dimension=CFG.embed_dim, max_input_tokens=512,
    normalize=True, batch_size=16,
)

## Configure the Oracle memory store

`OracleDBMemoryStore` is the durable storage and search implementation used by OAMP.

| Argument | Value | Meaning in this harness |
|---|---|---|
| `embedder` | `oamp_embedder` | Generates 384-dimensional vectors inside Oracle |
| `pool` | application pool | Supplies bounded independent database connections |
| `schema_policy` | `CREATE_IF_NECESSARY` | Creates package-owned objects on the first run and reuses them later |
| `memory_store_id` | `ERPACUSTOM` by default | Namespaces this harness from other agents in the same schema |
| `search_strategy` | `HYBRID` | Combines semantic similarity with lexical matching for IDs and exact terms |
| `search_index_sync` | `ON_COMMIT` | Makes committed memories searchable predictably for the lab |
| `memory_retention_config` | 90-day default, 365-day maximum | Bounds accidental indefinite retention while allowing explicit year-long records |

Hybrid retrieval matters for ERPA: vector similarity finds paraphrases, while lexical matching preserves exact identifiers such as `PO-BER-THC-OPEN`, SKU codes, and region names. `ON_COMMIT` is convenient for deterministic demonstrations; high-throughput deployments may prefer asynchronous index synchronization and explicit freshness objectives.

In [20]:

oamp_store = OracleDBMemoryStore(
    embedder=oamp_embedder, pool=pool,
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    memory_store_id=CFG.memory_store_id,
    search_strategy=SearchStrategy.HYBRID,
    search_index_sync=SearchIndexSyncMode.ON_COMMIT,
    memory_retention_config=MemoryRetentionConfig(default_ttl_days=90,max_ttl_days=365),
)


## Configure background extraction and context summaries

OAMP uses Anthropic through its `Llm` adapter for memory extraction. The main graph uses adaptive thinking; the extraction worker uses a fixed high reasoning effort and a smaller output budget because its job is structured distillation, not open-ended tool planning.

| Argument | Value | Operational effect |
|---|---|---|
| `extraction_mode` | `BACKGROUND` | Turn persistence can enqueue extraction instead of blocking every answer |
| queue-full behavior | `WAIT_THEN_RAISE` | Applies bounded backpressure; memory is not silently dropped |
| put timeout | 10 seconds | Prevents indefinite notebook stalls |
| `extract_memories` | `True` | Enables semantic fact/preference/guideline extraction |
| extraction frequency | every turn | Makes the teaching scenario observable quickly |
| extraction window | full eligible window | Lets related turns inform extraction within the token cap |
| token limit | 6000 | Bounds extraction context |
| context summary | enabled, update every two turns | Produces a compact context card for prompt assembly |
| custom instructions | merchandising policy | Preserves IDs, avoids causal invention, rejects secrets/raw PII |
| inherited metadata | tenant, session, source | Carries provenance into derived memories |

Background does not mean “fire and forget.” Acceptance cells call `wait_for_memory_extraction` before asserting new memories. A production service would expose queue depth, failures, age, retry policy, and deletion propagation as telemetry.

In [21]:
oamp_llm = Llm(model=f"anthropic/{CFG.model}", reasoning_effort="high", max_tokens=4096)
extraction_config = MemoryExtractionConfig(
    extraction_mode=MemoryExtractionMode.BACKGROUND,
    background_extraction_queue_full_behavior=BackgroundExtractionQueueFullBehavior.WAIT_THEN_RAISE,
    background_extraction_queue_put_timeout_seconds=10,
    extract_memories=True,
    memory_extraction_frequency=1,
    memory_extraction_window=-1,
    memory_extraction_token_limit=6000,
    enable_context_summary=True,
    context_summary_update_frequency=2,
    memory_extraction_custom_instructions=(
        "Extract explicit merchandising preferences, accepted decisions, product/region facts, "
        "and reusable operating guidance. Preserve SKUs, locations and purchase-order IDs verbatim. "
        "Do not infer causality from correlation. Never store secrets or raw PII."
    ),
    memory_extraction_inherit_message_metadata=("tenant_id","session_id","source"),
)


In [22]:

oamp = OracleAgentMemory(store=oamp_store,llm=oamp_llm,memory_extraction_config=extraction_config)

if oamp_store.get("user_profile", CFG.user_id) is None:
    oamp.add_user(CFG.user_id, "UK and EU regional merchandiser", metadata={"tenant_id": CFG.tenant_id})
if oamp_store.get("agent_profile", CFG.agent_id) is None:
    oamp.add_agent(CFG.agent_id, "Enterprise retail planning assistant", metadata={"tenant_id": CFG.tenant_id})

print({"memory": type(oamp).__name__, "oamp_version": OAMP_VERSION,
       "store": type(oamp_store).__name__, "search": "HYBRID",
       "embedder": CFG.embed_model, "llm": CFG.model})


{'memory': 'OracleAgentMemory', 'oamp_version': '26.6.0', 'store': 'OracleDBMemoryStore', 'search': 'HYBRID', 'embedder': 'ALL_MINILM_L12_V2', 'llm': 'claude-opus-4-8'}


## Wrap memory operations and promote scratch content with evidence

Jupyter already runs an event loop, while the OAMP API used here is synchronous. A one-worker executor moves those calls off the notebook's active asyncio thread without introducing concurrent memory ordering. `MemoryManager` then provides four harness-level operations: create/reuse a thread, add a turn with provenance, fetch a bounded context card, and write/search durable semantic records.

The wrapper is where application policy becomes concrete:

- every record is scoped to the configured user, agent, and tenant;
- thread messages carry session/source metadata;
- context cards request a balanced minimum of preferences, guidelines, and facts;
- semantic writes have a bounded TTL and distinguish explicit writes from scratch promotion;
- recall uses exact user/agent matching and hybrid search across approved record types.

Three deterministic seed memories establish a baseline operating scope, PO-suppression guideline, and brief-format preference. On a first run, the scratch verification token is asserted absent before promotion. On a resumed run after a late assertion, the cell instead verifies that the token already has a consumed queue row; this makes the evidence cell safely rerunnable without weakening provenance.

`drain_promotion_queue` scopes work to the session being ended, derives an idempotent OAMP memory ID from each Oracle queue ID, writes the chunk with scratch provenance, marks the queue consumed, and marks the source file promoted. Evidence is reconstructed from the current verification token rather than inferred from a global queue count, so a resumed notebook remains deterministic even when prior sessions have pending or consumed rows. The cell then displays two tables: consumed queue rows containing the original scratch chunk, and the actual records read back from OAMP, including database record ID, type, content, and metadata. Finally, semantic recall proves that the promoted PO identifier and unique token are discoverable from database memory.

This evidence is stronger than printing “promotion succeeded”: it connects source path → current chunk/token → queue ID → database-read durable memory record → recalled content.

In [23]:
from concurrent.futures import ThreadPoolExecutor

from langsmith import traceable
from oracleagentmemory.apis import Message


OAMP_EXECUTOR = ThreadPoolExecutor(max_workers=1, thread_name_prefix="erpa-oamp")


def oamp_call(function, *args, **kwargs):
    """Run OAMP's synchronous API outside Jupyter's active asyncio thread."""
    return OAMP_EXECUTOR.submit(function, *args, **kwargs).result()


class MemoryManager:
    def __init__(self, memory, store):
        self.memory = memory
        self.store = store

    def _thread(self, thread_id: str):
        try:
            return oamp_call(self.memory.get_thread, thread_id)
        except KeyError:
            return oamp_call(
                self.memory.create_thread,
                thread_id=thread_id, user_id=CFG.user_id, agent_id=CFG.agent_id,
                metadata={"tenant_id": CFG.tenant_id, "source": "standalone_notebook"},
                context_card_token_limit=6000,
            )

    @traceable(name="memory.write.turn", run_type="tool")
    def add_turn(self, thread_id: str, role: str, content: str, session_id: str):
        thread = self._thread(thread_id)
        return oamp_call(thread.add_messages, [Message(
            role=role, content=content,
            metadata={"tenant_id": CFG.tenant_id, "session_id": session_id, "source": "agent_turn"},
        )])

    @traceable(name="memory.read.context_card", run_type="retriever")
    def context_card(self, thread_id: str) -> str:
        try:
            thread = self._thread(thread_id)
            return oamp_call(
                thread.get_context_card,
                fallback_message_count=6, max_relevant_results=8, max_recent_messages=4,
                min_relevant_results_by_type={"preference": 1, "guideline": 1, "fact": 1},
            ).content
        except Exception:
            return ""

    @traceable(name="memory.write.semantic", run_type="tool")
    def remember(self, content: str, *, memory_type: str = "fact",
                 session_id: Optional[str] = None, memory_id: Optional[str] = None):
        kwargs = {
            "memory_type": memory_type, "user_id": CFG.user_id, "agent_id": CFG.agent_id,
            "metadata": {"tenant_id": CFG.tenant_id, "session_id": session_id,
                         "source": "scratch_promotion" if session_id else "explicit"},
            "ttl_days": 365,
        }
        if memory_id is not None:
            kwargs["memory_id"] = memory_id
        return oamp_call(self.memory.add_memory, content, **kwargs)

    @traceable(name="memory.read.semantic", run_type="retriever")
    def recall(self, query: str, limit: int = 8) -> list[str]:
        results = oamp_call(
            self.memory.search,
            query, user_id=CFG.user_id, agent_id=CFG.agent_id,
            exact_user_match=True, exact_agent_match=True, exact_thread_match=False,
            max_results=limit, record_types=["memory","guideline","fact","preference"],
            metadata_filter={"tenant_id": CFG.tenant_id},
        )
        return [getattr(getattr(item,"record",item),"content",getattr(item,"content","")) for item in results]


memory = MemoryManager(oamp, oamp_store)

seed_memories = [
    ("seed-erpa-fact-scope", "fact",
     "Alex owns the UK and EU regions and recurring Outerwear and Tops planning."),
    ("seed-erpa-guideline-po", "guideline",
     "Do not re-raise an inventory issue when an open purchase order already covers it."),
    ("seed-erpa-preference-format", "preference",
     "Morning briefs should lead with numeric actions and cite suppressed purchase orders."),
]
for memory_id, kind, text in seed_memories:
    if oamp_store.get(kind, memory_id) is None:
        oamp_call(
            oamp.add_memory,
            text, memory_type=kind, memory_id=memory_id,
            user_id=CFG.user_id, agent_id=CFG.agent_id,
            metadata={"tenant_id": CFG.tenant_id, "source": "notebook_seed"},
            ttl_days=365,
        )

before_promotion = memory.recall(SCRATCH_PROMOTION_TOKEN, 8)
token_already_promoted = any(SCRATCH_PROMOTION_TOKEN in item for item in before_promotion)
if token_already_promoted:
    prior_token_rows = fetch_all("""SELECT consumed FROM erpa_memory_promotion_queue
        WHERE session_id=:session_id AND DBMS_LOB.INSTR(chunk,:token)>0""",
        {"session_id": SCRATCH_DEMO_SESSION, "token": SCRATCH_PROMOTION_TOKEN})
    assert prior_token_rows and all(row["CONSUMED"] == "Y" for row in prior_token_rows)
else:
    # Hybrid retrieval may return related PO memories for an unseen random token.
    # Absence means no returned content contains this exact run token—not zero results.
    assert all(SCRATCH_PROMOTION_TOKEN not in item for item in before_promotion)


promotion_receipts: list[dict[str, Any]] = []


def _raw_hex(value: Any) -> str:
    return value.hex() if isinstance(value, (bytes, bytearray)) else str(value).replace("-", "")


def drain_promotion_queue(limit: int = 500, session_id: Optional[str] = None) -> int:
    rows = fetch_all("""SELECT queue_id,session_id,path,chunk FROM erpa_memory_promotion_queue
                        WHERE consumed='N'
                          AND (:session_id IS NULL OR session_id=:session_id)
                        ORDER BY staged_at FETCH FIRST :limit ROWS ONLY""",
                     {"session_id": session_id, "limit": limit})
    for row in rows:
        memory_id = f"scratch-{_raw_hex(row['QUEUE_ID']).lower()}"
        content = f"Promoted scratch note {row['PATH']}: {row['CHUNK']}"
        memory.remember(
            content, memory_type="fact", session_id=row["SESSION_ID"], memory_id=memory_id,
        )
        execute("""UPDATE erpa_memory_promotion_queue SET consumed='Y',consumed_at=SYSTIMESTAMP
                   WHERE queue_id=:queue_id""", {"queue_id": row["QUEUE_ID"]})
        promotion_receipts.append({
            "queue_id": _raw_hex(row["QUEUE_ID"]).upper(),
            "memory_id": memory_id, "memory_type": "fact", "session_id": row["SESSION_ID"],
            "scratch_path": row["PATH"], "scratch_chunk": str(row["CHUNK"]),
            "stored_content": content,
        })
    for session_id, path in {(row["SESSION_ID"], row["PATH"]) for row in rows}:
        execute("""UPDATE erpa_scratch_files SET promotion_state='Y'
                   WHERE session_id=:session_id AND path=:path""",
                {"session_id": session_id, "path": path})
    return len(rows)


newly_promoted = drain_promotion_queue(session_id=SCRATCH_DEMO_SESSION)
oamp_call(oamp.wait_for_memory_extraction, timeout=300)
recalled = memory.recall("Which purchase order covers Berlin ThermaCore?", 5)
assert any("PO-BER-THC-OPEN" in item for item in recalled)
promoted_token = memory.recall(SCRATCH_PROMOTION_TOKEN, 8)
assert any(SCRATCH_PROMOTION_TOKEN in item for item in promoted_token)

# Show both sides of the durable handoff: the consumed queue row and the exact
# OAMP fact IDs/content written from those scratch chunks.
promotion_queue_evidence = fetch_all("""SELECT RAWTOHEX(queue_id) queue_id,session_id,path,
    DBMS_LOB.SUBSTR(chunk,500,1) scratch_chunk,consumed,consumed_at
    FROM erpa_memory_promotion_queue
    WHERE session_id=:session_id AND DBMS_LOB.INSTR(chunk,:token)>0
    ORDER BY staged_at""",
    {"session_id": SCRATCH_DEMO_SESSION, "token": SCRATCH_PROMOTION_TOKEN})
assert promotion_queue_evidence
assert all(row["CONSUMED"] == "Y" for row in promotion_queue_evidence)
promoted_db_entries = []
for queue_row in promotion_queue_evidence:
    memory_id = f"scratch-{queue_row['QUEUE_ID'].lower()}"
    record = oamp_store.get("fact", memory_id)
    if record is not None:
        promoted_db_entries.append({
            "queue_id": queue_row["QUEUE_ID"],
            "memory_id": record.id,
            "record_type": record.record_type,
            "database_content": record.content,
            "database_metadata": json.dumps(record.metadata, default=str, sort_keys=True),
        })
promoted = len(promotion_queue_evidence)
assert len(promoted_db_entries) == promoted
display(pd.DataFrame(promotion_queue_evidence))
display(pd.DataFrame(promoted_db_entries))
print({"verified_token_chunks": promoted, "newly_promoted_this_run": newly_promoted,
       "database_memory_rows": len(promoted_db_entries), "recall": recalled[:3],
       "session_token_promoted": True, "resumed_after_prior_promotion": token_already_promoted})

,QUEUE_ID,SESSION_ID,PATH,SCRATCH_CHUNK,CONSUMED,CONSUMED_AT
0,58DA5DA4236F0F82E063030011ACFE22,scratch-demo,/scratch/notes/supplier.md,Berlin ThermaCore is covered by PO-BER-THC-OPE...,Y,2026-08-12 13:14:48.090610


,queue_id,memory_id,record_type,database_content,database_metadata
0,58DA5DA4236F0F82E063030011ACFE22,scratch-58da5da4236f0f82e063030011acfe22,fact,Promoted scratch note /scratch/notes/supplier....,"{""session_id"": ""scratch-demo"", ""source"": ""scra..."


{'verified_token_chunks': 1, 'newly_promoted_this_run': 2, 'database_memory_rows': 1, 'recall': ['Promoted scratch note /scratch/notes/supplier.md: Berlin ThermaCore is covered by PO-BER-THC-OPEN. Verification token: promotion-91b4052fc7a04c02ba73fd39d95b0a01.', 'Promoted scratch note /scratch/notes/supplier.md: Berlin ThermaCore is covered by PO-BER-THC-OPEN. Verification token: promotion-1bbbfeb79ff645849ddd52f490930d8a.', 'Promoted scratch note /scratch/notes/supplier.md: Berlin ThermaCore is covered by PO-BER-THC-OPEN. Verification token: promotion-307a2c155dc94d37b6fb505ce2f8c6f6.'], 'session_token_promoted': True, 'resumed_after_prior_promotion': False}


Background work item failed.


# Part 6 · Progressive toolbox and skill disclosure

Tools are capabilities; skills are approved procedures for combining capabilities. Dumping every schema and full procedure into every prompt wastes context and weakens attention. ERPA stores compact, embedded registries in Oracle and discloses them progressively:

1. vector-search the current request against short tool descriptions and skill manifests;
2. bind only the selected tool schemas to Claude;
3. place only short skill manifests in assembled context; and
4. require `load_skill` to retrieve a matching full procedure before execution.

The registry contains trusted Oracle data tools, ScratchFS operations, durable memory, skill loading, and the E2B sandbox. The curated skills cover morning briefs, spike diagnosis, size-curve review, profitability ranking, evidence triangulation, scratch planning, preference capture, and approval preparation. Hashes make skill bodies inspectable for change.

This is capability minimization, not merely token optimization. A model cannot call an unbound tool, and `CustomToolbox` still checks an in-process allowlist at dispatch time.

In [24]:
ddl("""CREATE TABLE erpa_tool_registry (
    tool_name VARCHAR2(80) PRIMARY KEY, description VARCHAR2(1000),
    transport VARCHAR2(80), embedding VECTOR(384,FLOAT32))""")
ddl("""CREATE TABLE erpa_skill_registry (
    skill_name VARCHAR2(80) PRIMARY KEY, description VARCHAR2(1000), body CLOB,
    body_sha256 VARCHAR2(64), status VARCHAR2(20), embedding VECTOR(384,FLOAT32))""")

tool_rows = [
    ("morning_brief_inputs", "Find new restock actions and suppress open purchase orders", "trusted Oracle SQL"),
    ("sales_signal", "Compare current weekly sales with prior weeks and promotions", "trusted Oracle SQL"),
    ("inventory_status", "Read inventory by product, region and size", "trusted Oracle SQL"),
    ("regional_profitability", "Rank regions by governed gross margin", "trusted Oracle semantic view"),
    ("institutional_search", "Search company guidance and review notes", "Oracle vector search"),
    ("external_signal_search", "Read dated external market and weather signals", "trusted Oracle feed"),
    ("scratch_write", "Write or update the session plan and short-term notes", "Oracle SecureFile ScratchFS"),
    ("scratch_read", "Read a session scratch file", "Oracle SecureFile ScratchFS"),
    ("sandbox_python", "Execute generated Python or chart code in E2B", "E2B Code Interpreter"),
    ("remember_fact", "Persist a durable user preference or accepted decision", "Oracle Agent Memory"),
    ("load_skill", "Load the full body of a retrieved skill", "Oracle skill registry"),
]
skills = [
    ("morning-brief", "Prepare Alex's numeric morning attention brief",
     "Query morning_brief_inputs. Report new actions first. Suppress any row covered by an OPEN purchase order and cite its PO ID. Use remembered recurring scope and exclusions."),
    ("spike-investigation", "Explain a product or regional sales spike",
     "Triangulate internal weekly sales, internal promotion flags, institutional guidance, and one dated external signal. State correlation as a hypothesis, not proof."),
    ("inventory-visual", "Visualise product inventory by region and size",
     "Query inventory_status, then use sandbox_python to create an SVG or compact statistical summary. Store large output in ScratchFS and return its pointer."),
    ("profitability", "Compare regional commercial profitability",
     "Use regional_profitability. Profitability means governed gross_margin, not revenue. Explain discounting when it changes the ranking."),
]

with connect() as con:
    cur = con.cursor()
    cur.execute("DELETE FROM erpa_tool_registry")
    cur.execute("DELETE FROM erpa_skill_registry")
    tool_binds = [
        {"name": name, "description": description, "transport": transport}
        for name, description, transport in tool_rows
    ]
    cur.executemany(
        f"""INSERT INTO erpa_tool_registry(tool_name,description,transport,embedding)
        VALUES (:name,:description,:transport,
          VECTOR_EMBEDDING({CFG.embed_model} USING :description AS DATA))""",
        tool_binds,
    )
    skill_binds = [
        {"name": name, "description": description, "body": body,
         "body_sha256": hashlib.sha256(body.encode()).hexdigest(), "status": "ACTIVE"}
        for name, description, body in skills
    ]
    cur.executemany(
        f"""INSERT INTO erpa_skill_registry
          (skill_name,description,body,body_sha256,status,embedding)
        VALUES (:name,:description,:body,:body_sha256,:status,
          VECTOR_EMBEDDING({CFG.embed_model} USING :description AS DATA))""",
        skill_binds,
    )
    con.commit()


def retrieve_tool_names(query: str, k: int = 7) -> list[str]:
    rows = fetch_all(f"""SELECT tool_name,
        VECTOR_DISTANCE(embedding,VECTOR_EMBEDDING({CFG.embed_model} USING :query AS DATA),COSINE) distance
        FROM erpa_tool_registry ORDER BY distance FETCH APPROX FIRST :k ROWS ONLY""",
        {"query": query, "k": k})
    essentials = ["scratch_write", "scratch_read", "remember_fact", "load_skill"]
    return list(dict.fromkeys([row["TOOL_NAME"] for row in rows] + essentials))


def retrieve_skill_manifest(query: str, k: int = 3) -> str:
    rows = fetch_all(f"""SELECT skill_name,description,
        VECTOR_DISTANCE(embedding,VECTOR_EMBEDDING({CFG.embed_model} USING :query AS DATA),COSINE) distance
        FROM erpa_skill_registry WHERE status='ACTIVE'
        ORDER BY distance FETCH APPROX FIRST :k ROWS ONLY""", {"query": query, "k": k})
    return "\n".join(f"- {r['SKILL_NAME']}: {r['DESCRIPTION']} (distance={r['DISTANCE']:.3f})" for r in rows)


print("Tools:", retrieve_tool_names("Why did WarmLayer spike in the UK?"))
print("Skills:\n", retrieve_skill_manifest("Prepare my morning brief"))


Tools: ['external_signal_search', 'sales_signal', 'sandbox_python', 'regional_profitability', 'morning_brief_inputs', 'institutional_search', 'inventory_status', 'scratch_write', 'scratch_read', 'remember_fact', 'load_skill']
Skills:
 - morning-brief: Prepare Alex's numeric morning attention brief (distance=0.398)
- spike-investigation: Explain a product or regional sales spike (distance=0.985)
- profitability: Compare regional commercial profitability (distance=0.989)


# Part 7 · Trusted function tools and the Custom Toolbox

These functions are trusted application code. They use bound parameters, return JSON-safe values, and expose narrow business capabilities. The model never receives a general SQL or shell tool and never sees database credentials. Generated Python has one route—E2B—and fails closed if that provider is unavailable.

| Tool | Kind | Reads or changes | Returned evidence | Safety boundary |
|---|---|---|---|---|
| `morning_brief_inputs` | Oracle function | Reads low stock and open POs | New actions plus handled suppressions | Fixed joins and filters; no arbitrary SQL |
| `inventory_status` | Oracle function | Reads stock by product | Region/city/size on-hand and reorder values | Bound product filter |
| `sales_signal` | Oracle function | Reads weekly sales | Current units, baseline, change and promotion flag | Bound product/region scope |
| `regional_profitability` | Semantic-view function | Reads governed commercial facts | Gross-margin value, percentage and discounts | Metric fixed in `ERPA_V_PROFITABILITY` |
| `institutional_search` | Oracle vector retriever | Reads approved internal notes | Text, source and cosine distance | Curated corpus only |
| `external_signal_search` | Oracle function | Reads ingested outside signals | Date, observation and source URL | Governed ingestion table |
| `scratch_write` / `scratch_read` | SecureFile functions | Changes or reads session scratch | Scoped path, size and content | Cannot escape the session mount |
| `sandbox_python` | E2B function | Runs generated computation | E2B logs/result or ScratchFS pointer | No host execution fallback |
| `remember_fact` | OAMP function | Writes durable memory | OAMP memory identifier | Explicit type allowlist |
| `load_skill` | Oracle function | Reads one approved procedure | Body plus content SHA | `ACTIVE` skills only |

The **Custom Toolbox** joins two stores without confusing their roles:

- Oracle persists deterministic tool metadata and embeddings for semantic discovery.
- The trusted Python process retains the callable implementations and credentials.
- A query reveals only the smallest relevant schema set.
- Execution accepts only a registered name and lets each LangChain tool's Pydantic schema validate arguments.

```mermaid
flowchart LR
    F[Trusted Python function] --> S[LangChain tool schema]
    M[Description + transport] --> E[Oracle in-DB embedding]
    E --> DB[(ERPA_TOOL_REGISTRY)]
    Q[Current intent] --> V[Vector discovery]
    DB --> V
    V --> D[Disclose selected schemas]
    D --> L{Allowlisted name?}
    L -->|yes| P[Pydantic argument validation]
    P --> F
    L -->|no| X[Reject]
```

> 🔩 **Harness connection / decision**
>
> tool schemas are an authority boundary. The model receives capabilities, not infrastructure credentials; semantic retrieval changes disclosure, never execution authority.


### Trust model

The word “tool” does not grant arbitrary authority. Trusted SQL is fixed and bind-variable based; semantic retrieval returns text and provenance but cannot mutate data; ScratchFS is session/mount scoped; OAMP writes accept only approved memory types; generated Python has one E2B transport; and unknown tool names return `tool_not_allowed`.

The E2B tool creates a fresh sandbox, executes code remotely, captures structured logs/results, and always kills the sandbox. Large serialized results are written to ScratchFS and replaced in model context with a path and preview. The smoke assertion proves the provider is E2B—there is deliberately no local `exec`, `eval`, subprocess, or host-Python fallback for model-generated code.

In [25]:
from e2b_code_interpreter import Sandbox
from langchain_core.tools import tool


def _json(value: Any) -> str:
    return json.dumps(value, default=str)


@tool
@traceable(name="tool.morning_brief_inputs", run_type="tool")
def morning_brief_inputs(excluded_category: str = "") -> str:
    """Return new UK/EU restock actions and issues suppressed by an open purchase order."""
    rows = fetch_all("""SELECT i.sku,i.product_line,i.category,i.region,i.city,i.size_code,
        i.on_hand,i.reorder_point,p.po_id,p.status
        FROM erpa_inventory i LEFT JOIN erpa_purchase_orders p ON p.sku=i.sku AND p.status='OPEN'
        WHERE i.region IN ('UK','EU') AND i.category IN ('Outerwear','Tops')
          AND i.on_hand<i.reorder_point
          AND (:excluded IS NULL OR UPPER(i.category)<>UPPER(:excluded))
        ORDER BY i.region,i.city,i.product_line""", {"excluded": excluded_category or None})
    suppressed = [row for row in rows if row["PO_ID"]]
    actions = [row for row in rows if not row["PO_ID"]]
    return _json({"new_actions": actions, "suppressed": suppressed, "as_of": str(date.today())})


@tool
@traceable(name="tool.sales_signal", run_type="tool")
def sales_signal(product_line: str, region: str) -> str:
    """Compare the latest weekly units with the preceding observations."""
    rows = fetch_all("""SELECT week_start,units,promotion_flag FROM erpa_weekly_sales
        WHERE UPPER(product_line)=UPPER(:product) AND UPPER(region)=UPPER(:region)
        ORDER BY week_start""", {"product": product_line, "region": region})
    if len(rows) < 2:
        return _json({"error": "not enough observations"})
    current = rows[-1]
    average = sum(row["UNITS"] for row in rows[:-1]) / len(rows[:-1])
    return _json({"product_line": product_line, "region": region,
                  "current_units": current["UNITS"], "prior_weekly_average": average,
                  "change_percent": round((current["UNITS"] / average - 1) * 100),
                  "internal_promotion": current["PROMOTION_FLAG"] == "Y"})


@tool
@traceable(name="tool.inventory_status", run_type="tool")
def inventory_status(product_line: str) -> str:
    """Return inventory rows by product, region, city and size."""
    return _json(fetch_all("""SELECT sku,region,city,size_code,on_hand,reorder_point
        FROM erpa_inventory WHERE UPPER(product_line)=UPPER(:product)
        ORDER BY region,city,size_code""", {"product": product_line}))


@tool
@traceable(name="tool.regional_profitability", run_type="tool")
def regional_profitability() -> str:
    """Rank regions using the governed gross-margin definition of profitability."""
    return _json(fetch_all("""SELECT region,units,net_revenue,gross_margin,
        gross_margin_percent,avg_discount_percent FROM erpa_v_profitability
        ORDER BY gross_margin DESC"""))


@tool
@traceable(name="tool.institutional_search", run_type="retriever")
def institutional_search(query: str, limit: int = 3) -> str:
    """Search trusted internal notes by meaning."""
    return _json(fetch_all(f"""SELECT title,body,source_url,
        VECTOR_DISTANCE(embedding,VECTOR_EMBEDDING({CFG.embed_model} USING :query AS DATA),COSINE) distance
        FROM erpa_institutional_docs ORDER BY distance FETCH APPROX FIRST :limit ROWS ONLY""",
        {"query": query, "limit": limit}))


@tool
@traceable(name="tool.external_signal_search", run_type="retriever")
def external_signal_search(region: str, signal_type: str = "weather") -> str:
    """Read dated external signals from the governed ingestion table."""
    return _json(fetch_all("""SELECT observed_at,region,signal_type,summary,source_url
        FROM erpa_external_signals WHERE UPPER(region)=UPPER(:region)
          AND UPPER(signal_type)=UPPER(:kind) ORDER BY observed_at DESC""",
        {"region": region, "kind": signal_type}))


def _session_fs(session_id: str) -> ScratchFS:
    if not fetch_all("SELECT 1 FROM erpa_agent_sessions WHERE session_id=:s", {"s": session_id}):
        raise ValueError("Unknown session_id; begin the session before using scratch tools")
    return ScratchFS(session_id)


@tool
@traceable(name="tool.scratch_write", run_type="tool")
def scratch_write(session_id: str, path: str, content: str, promote_on_end: bool = False) -> str:
    """Write a plan, note, draft, or tool artifact to Oracle ScratchFS."""
    return _json(_session_fs(session_id).write(path, content, promote_on_end=promote_on_end))


@tool
@traceable(name="tool.scratch_read", run_type="tool")
def scratch_read(session_id: str, path: str) -> str:
    """Read a file from the current session's Oracle ScratchFS."""
    return _session_fs(session_id).read(path)


@tool
@traceable(name="tool.remember_fact", run_type="tool")
def remember_fact(content: str, memory_type: str = "preference") -> str:
    """Persist an explicit user preference, fact, or reusable guideline with OAMP."""
    if memory_type not in {"preference", "fact", "guideline"}:
        raise ValueError("memory_type must be preference, fact, or guideline")
    return _json({"memory_id": str(memory.remember(content, memory_type=memory_type))})


@tool
@traceable(name="tool.load_skill", run_type="retriever")
def load_skill(skill_name: str) -> str:
    """Load the full body of one retrieved active skill."""
    rows = fetch_all("SELECT body,body_sha256 FROM erpa_skill_registry WHERE skill_name=:name AND status='ACTIVE'",
                     {"name": skill_name})
    return _json(rows[0] if rows else {"error": "skill not found"})


@tool
@traceable(name="tool.sandbox_python", run_type="tool")
def sandbox_python(session_id: str, code: str, artifact_path: str = "/tool_out/result.txt") -> str:
    """Execute generated Python only in E2B; store long output in Oracle ScratchFS."""
    sandbox = Sandbox.create(api_key=E2B_API_KEY, timeout=120)
    try:
        execution = sandbox.run_code(code)
        stdout = "\n".join(str(item) for item in execution.logs.stdout)
        stderr = "\n".join(str(item) for item in execution.logs.stderr)
        payload = json.loads(execution.to_json())
        payload.update({"provider": "e2b", "stdout": stdout, "stderr": stderr})
    finally:
        sandbox.kill()
    serialised = _json(payload)
    if len(serialised) > 3000:
        pointer = _session_fs(session_id).write(artifact_path, serialised)
        return _json({"provider": "e2b", "file_pointer": pointer["path"],
                      "preview": serialised[:800]})
    return serialised


TOOLS = {item.name: item for item in [
    morning_brief_inputs, sales_signal, inventory_status, regional_profitability,
    institutional_search, external_signal_search, scratch_write, scratch_read,
    sandbox_python, remember_fact, load_skill,
]}

smoke = json.loads(sandbox_python.invoke({
    "session_id": SCRATCH_DEMO_SESSION,
    "code": "values=[8,5,6]; print({'restock_units': sum(values)})",
}))
assert smoke["provider"] == "e2b" and not smoke["error"]
print(smoke)


{'results': [], 'logs': '{"stdout": ["{\'restock_units\': 19}\\n"], "stderr": []}', 'error': None, 'provider': 'e2b', 'stdout': "{'restock_units': 19}\n", 'stderr': ''}


In [26]:
class CustomToolbox:
    """Oracle-discovered metadata plus an in-process allowlisted executor."""
    def __init__(self, functions: dict[str, Any]):
        self._functions = dict(functions)

    @traceable(name="toolbox.discover", run_type="retriever")
    def discover(self, query: str, limit: int = 7) -> dict[str, Any]:
        names = retrieve_tool_names(query, limit)
        # The registry is intentionally small. Read it with one fixed query and
        # filter in trusted Python rather than constructing a dynamic SQL IN list.
        registry_rows = fetch_all(
            "SELECT tool_name,description,transport FROM erpa_tool_registry"
        )
        metadata = {row["TOOL_NAME"]: row for row in registry_rows}
        tools = []
        for name in names:
            function = self._functions.get(name)
            if function is None:
                continue
            row = metadata[name]
            tools.append({
                "name": name,
                "description": row["DESCRIPTION"],
                "transport": row["TRANSPORT"],
                "parameters": function.args,
            })
        return {"query": query, "tools": tools}

    def discover_names(self, query: str, limit: int = 7) -> list[str]:
        return [item["name"] for item in self.discover(query, limit)["tools"]]

    def tools_for_names(self, names: list[str]) -> list[Any]:
        return [self._functions[name] for name in names if name in self._functions]

    @traceable(name="toolbox.invoke", run_type="tool")
    def invoke(self, tool_name: str, arguments: dict[str, Any] | None = None) -> str:
        function = self._functions.get(tool_name)
        if function is None:
            return _json({"error": "tool_not_allowed", "tool_name": tool_name})
        try:
            return str(function.invoke(arguments or {}))
        except Exception as exc:
            return _json({
                "error": "invalid_or_failed_tool_call",
                "tool_name": tool_name,
                "error_type": type(exc).__name__,
                "detail": str(exc),
                "allowed_parameters": function.args,
            })


In [27]:
custom_toolbox = CustomToolbox(TOOLS)
discovered = custom_toolbox.discover("Prepare a morning stock brief", limit=5)
denied = json.loads(custom_toolbox.invoke("run_arbitrary_sql", {"sql": "DROP TABLE anything"}))
assert discovered["tools"] and denied["error"] == "tool_not_allowed"
print({"discovered": [item["name"] for item in discovered["tools"]], "denied": denied})

{'discovered': ['inventory_status', 'scratch_write', 'morning_brief_inputs', 'institutional_search', 'external_signal_search', 'scratch_read', 'remember_fact', 'load_skill'], 'denied': {'error': 'tool_not_allowed', 'tool_name': 'run_arbitrary_sql'}}


# Part 8 · Claude Opus 4.8 in adaptive-thinking mode

Claude is the stateless reasoner at the center of the harness. `thinking={"type": "adaptive"}` lets the model decide when a request benefits from deeper internal reasoning. A larger `max_tokens` budget accommodates thinking and a concise final response, but budgets in LangGraph still bound iterations and wall-clock time.

The bare invocation is an intentional control experiment. It proves the model adapter works, but it also demonstrates what is missing: no Oracle facts, no memory, no semantic definitions, no scratch plan, no skills, no E2B, no checkpoints, and no durable effects. Those capabilities appear only when the graph assembles and governs them.

Anthropic is the only chat/reasoning provider used by this notebook. Oracle's ONNX model is an embedder, not a second answer-generating model.

In [28]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

llm = ChatAnthropic(
    model=CFG.model,
    api_key=ANTHROPIC_API_KEY,
    max_tokens=8192,
    thinking={"type": "adaptive"},
)
bare = llm.invoke([HumanMessage(content="In one sentence, explain why a model is not an agent harness.")])
assert bare.content
print({"model": CFG.model, "thinking": "adaptive", "response_blocks": len(bare.content) if isinstance(bare.content,list) else 1})


{'model': 'claude-opus-4-8', 'thinking': 'adaptive', 'response_blocks': 1}


## Prompt caching: where it would fit (not implemented)

Anthropic prompt caching and Oracle semantic caching solve different problems. Prompt caching can reduce repeated input processing for an identical stable prefix; semantic caching reuses a completed answer for meaning-equivalent eligible questions.

A future prompt-caching layout would put the stable ERPA role, safety rules, tool contracts, and approved skill authority first and mark that prefix with Anthropic cache control. Volatile OAMP memories, semantic matches, scratch plans, tool results, and the current message would follow outside the stable boundary. Any policy/tool-schema change should intentionally invalidate the prefix.

This notebook does **not** set `cache_control`. The omission is deliberate and visible in the component map. It keeps the exercise focused on state and execution boundaries while documenting the correct optimization seam. Part 11 does implement Oracle semantic caching and measures graph bypass directly.

# Part 9 · LangGraph orchestration with OracleSaver

The orchestration graph makes the agent loop explicit: assemble selective context, call the model with a retrieved allowlist, dispatch requested tools, repeat with new evidence, and persist the completed turn. Conditional edges keep this loop inspectable and impose iteration and wall-clock budgets.

`OracleSaver` is durable computational state, not long-term user memory. It stores graph checkpoints at super-step boundaries under `configurable.thread_id`; reusing a thread ID deliberately continues that timeline. OAMP separately provides cross-thread memory. LangSmith wraps the nodes and tools so the control path can be inspected without querying checkpoint internals.

| Node | Reads | Writes | Primary control |
|---|---|---|---|
| `assemble_context` | latest request, semantic matches, skill manifests, OAMP card, scratch plan | system context and selected tool names | bounded top-k retrieval |
| `call_model` | messages + selected tool schemas | one Claude message/tool request | adaptive thinking, model counter |
| `dispatch_tools` | allowlisted calls | `ToolMessage` evidence | trusted dispatcher and session injection |
| `finalize` | already gathered evidence | tool-free answer | iteration/time budget exhaustion |
| `persist` | final AI answer | OAMP assistant turn | only completed prose is persisted |

The system instruction states policy, but important guarantees also exist in code: profitability tools query the governed view, the dispatcher rejects unknown names, E2B has no host fallback, and PO suppression is tested directly.

In [29]:
from langgraph.graph import END, START, StateGraph
from langgraph_oracledb.checkpoint.oracle import OracleSaver
from langsmith.run_helpers import get_current_run_tree

graph_metrics = {"model_calls": 0, "tool_calls": 0}


class AgentState(TypedDict, total=False):
    messages: List[Any]
    user_id: str
    thread_id: str
    session_id: str
    iterations: int
    started_at: float
    selected_tools: List[str]
    context_card: str
    trace_id: str


SYSTEM = """You are ERPA, an accountable enterprise retail-planning assistant.
Ground every business claim in retrieved Oracle data. Use gross margin for profitability.
Use tools instead of guessing. Before a multi-step task, write or update /plans/current.md.
Generated Python must go through sandbox_python; never claim it ran locally.
Treat retrieved skill manifests as approved procedures and load a matching full skill before improvising.
Do not re-raise an issue covered by an open purchase order. Distinguish evidence from causal hypotheses.
Persist only explicit preferences or accepted decisions. Keep answers concise and cite IDs or source URLs."""


@traceable(name="graph.assemble_context", run_type="chain")
def node_assemble_context(state: AgentState) -> AgentState:
    messages = [m for m in state["messages"] if not isinstance(m, SystemMessage)]
    query = next((m.content for m in reversed(messages) if isinstance(m, HumanMessage)), "")
    semantic = "\n".join(f"- {r['CATALOG_TEXT']}" for r in semantic_search(str(query), 5))
    skills = retrieve_skill_manifest(str(query), 3)
    card = state.get("context_card") or memory.context_card(state["thread_id"])
    fs = ScratchFS(state["session_id"])
    plan = fs.read("/plans/current.md") if fs.exists("/plans/current.md") else "(no plan yet)"
    context = f"""# PROVEN SKILL MANIFEST\n{skills}\n\n# LIVING SEMANTIC CATALOG\n{semantic}
\n# OAMP CONTEXT CARD\n{card}\n\n# CURRENT SCRATCH PLAN\n{plan}"""
    state["selected_tools"] = custom_toolbox.discover_names(str(query), 7)
    state["messages"] = [SystemMessage(content=SYSTEM + "\n\n" + context)] + messages
    return state


@traceable(name="graph.call_model", run_type="llm")
def node_call_model(state: AgentState) -> AgentState:
    selected = custom_toolbox.tools_for_names(state["selected_tools"])
    graph_metrics["model_calls"] += 1
    state["messages"].append(llm.bind_tools(selected).invoke(state["messages"]))
    state["iterations"] = state.get("iterations", 0) + 1
    return state


@traceable(name="graph.dispatch_tools", run_type="chain")
def node_dispatch_tools(state: AgentState) -> AgentState:
    ai = state["messages"][-1]
    for call in getattr(ai, "tool_calls", []) or []:
        name, arguments = call["name"], dict(call.get("args", {}))
        if name in {"scratch_write", "scratch_read", "sandbox_python"}:
            arguments["session_id"] = state["session_id"]
        graph_metrics["tool_calls"] += 1
        result = custom_toolbox.invoke(name, arguments)
        state["messages"].append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return state


@traceable(name="graph.persist", run_type="chain")
def node_persist(state: AgentState) -> AgentState:
    final = next((m for m in reversed(state["messages"]) if isinstance(m, AIMessage) and m.content), None)
    if final:
        content = _message_text(final)
        memory.add_turn(state["thread_id"], "assistant", content, state["session_id"])
    return state


@traceable(name="graph.finalize", run_type="chain")
def node_finalize(state: AgentState) -> AgentState:
    """Force one tool-free closing answer if a run reaches its budget mid-tool-use."""
    messages = [m for m in state["messages"]
                if not (isinstance(m, AIMessage) and getattr(m, "tool_calls", None) and not _message_text(m))]
    messages.append(HumanMessage(content=(
        "The tool budget is exhausted. Answer now using only the evidence already gathered. "
        "State any unresolved limitation explicitly."
    )))
    state["messages"].append(llm.invoke(messages))
    return state


def route_after_model(state: AgentState) -> str:
    ai = state["messages"][-1]
    wants_tools = bool(getattr(ai, "tool_calls", None))
    over_budget = state.get("iterations", 0) >= 8 or time.monotonic() - state["started_at"] > 180
    if over_budget and wants_tools:
        return "finalize"
    return "persist" if not wants_tools else "tools"



## Wire the control-flow graph and durable checkpoint store

The graph begins at `assemble_context`, always calls Claude once, and then branches on the presence of tool calls. A normal answer goes to `persist`; tool requests go through `dispatch_tools` and return to context assembly so new observations can inform the next decision. If the model still requests tools after the iteration or time budget, `finalize` asks for a tool-free answer from evidence already gathered. Both completion paths persist before `END`.

```mermaid
flowchart LR
    S((START)) --> A[assemble_context]
    A --> M[call_model]
    M -->|tool calls| T[dispatch_tools]
    T --> A
    M -->|answer| P[persist]
    M -->|budget + tool calls| F[finalize]
    F --> P
    P --> E((END))
```

The dedicated checkpoint connection isolates graph writes from business-tool and OAMP transactions. `OracleSaver.setup()` creates its package tables idempotently; `compile(checkpointer=checkpointer)` activates durable state. Rendering Mermaid from `graph.get_graph()` provides an implementation-derived diagram, so documentation and actual edges cannot drift silently.

In [30]:
builder = StateGraph(AgentState)
builder.add_node("assemble_context", node_assemble_context)
builder.add_node("call_model", node_call_model)
builder.add_node("dispatch_tools", node_dispatch_tools)
builder.add_node("persist", node_persist)
builder.add_node("finalize", node_finalize)
builder.add_edge(START, "assemble_context")
builder.add_edge("assemble_context", "call_model")
builder.add_conditional_edges(
    "call_model", route_after_model,
    {"tools": "dispatch_tools", "persist": "persist", "finalize": "finalize"},
)
builder.add_edge("dispatch_tools", "assemble_context")
builder.add_edge("finalize", "persist")
builder.add_edge("persist", END)



In [31]:

# A dedicated connection isolates checkpoint writes from OAMP/tool transactions.
checkpoint_connection = connect()
checkpointer = OracleSaver(checkpoint_connection, json_size_threshold_mb=0.0)
checkpointer.setup()

In [32]:
graph = builder.compile(checkpointer=checkpointer)
graph.get_graph().draw_mermaid()

'---\nconfig:\n  flowchart:\n    curve: linear\n---\ngraph TD;\n\t__start__([<p>__start__</p>]):::first\n\tassemble_context(assemble_context)\n\tcall_model(call_model)\n\tdispatch_tools(dispatch_tools)\n\tpersist(persist)\n\tfinalize(finalize)\n\t__end__([<p>__end__</p>]):::last\n\t__start__ --> assemble_context;\n\tassemble_context --> call_model;\n\tcall_model -. &nbsp;tools&nbsp; .-> dispatch_tools;\n\tcall_model -.-> finalize;\n\tcall_model -.-> persist;\n\tdispatch_tools --> assemble_context;\n\tfinalize --> persist;\n\tpersist --> __end__;\n\tclassDef default fill:#f2f0ff,line-height:1.2\n\tclassDef first fill-opacity:0\n\tclassDef last fill:#bfb6fc\n'

In [33]:
def _message_text(message: AIMessage) -> str:
    if isinstance(message.content, str):
        return message.content
    blocks = message.content if isinstance(message.content, list) else [message.content]
    return "\n".join(str(block.get("text", "")) if isinstance(block, dict) else str(block)
                     for block in blocks if not isinstance(block, dict) or block.get("type") == "text").strip()


## Expose one governed run boundary

`run_agent` is the application-facing entry point. It composes cache policy, session lifecycle, scratch planning, OAMP turns, LangGraph configuration, tracing metadata, answer extraction, and cache update in one testable boundary.

The order is deliberate:

1. classify cache eligibility and build a tenant/user/model/data-scoped namespace;
2. return immediately on a semantic-cache hit, before creating graph/tool/checkpoint work;
3. begin the logical session and create a current plan if absent;
4. persist the user turn and build initial graph state;
5. invoke or stream the graph under a stable `thread_id` and recursion limit;
6. extract the final textual AI message;
7. update the semantic cache only after a successful eligible miss; and
8. return the answer plus cache and LangSmith evidence.

The five-turn scenario uses `bypass_cache=True` when a live seam must be exercised. `end_session` performs the logical state transition, drains scratch promotion, and waits for background extraction before returning.

One subtlety: a cache hit has no new OracleSaver checkpoint and no new OAMP turn by design. The cache measurement later proves this with model/tool counters. If conversation bookkeeping is required even on hits, persist a separate audit event without pretending the full graph ran.

In [34]:

@traceable(name="erpa.run", run_type="chain")
def run_agent(text: str, *, session_id: str, thread_id: str, user_id: str = CFG.user_id,
              stream: bool = False, bypass_cache: bool = False) -> dict[str, Any]:
    cacheable = not bypass_cache and is_semantic_cache_eligible(text)
    cache_prompt = scoped_cache_prompt(text, user_id)
    namespace = semantic_cache_namespace(user_id)
    if cacheable:
        cached = semantic_cache.lookup(cache_prompt, namespace)
        if cached is not None:
            cache_metrics["hits"] += 1
            cache_events.append({"status": "HIT", "question": text, "namespace": namespace})
            run_tree = get_current_run_tree()
            return {
                "answer": cached[0].text,
                "state": None,
                "thread_id": thread_id,
                "session_id": session_id,
                "cache_hit": True,
                "langsmith_run_id": str(run_tree.id) if run_tree else None,
            }
        cache_metrics["misses"] += 1
        cache_events.append({"status": "MISS", "question": text, "namespace": namespace})

    begin_session(session_id, thread_id, user_id)
    fs = ScratchFS(session_id)
    if not fs.exists("/plans/current.md"):
        fs.write("/plans/current.md", f"# Current objective\n{text}\n")
    memory.add_turn(thread_id, "user", text, session_id)
    state: AgentState = {
        "messages": [HumanMessage(content=text)], "user_id": user_id,
        "thread_id": thread_id, "session_id": session_id,
        "iterations": 0, "started_at": time.monotonic(),
        "context_card": memory.context_card(thread_id),
    }
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 50,
              "tags": ["erpa", "standalone-notebook"],
              "metadata": {"session_id": session_id, "user_id": user_id, "model": CFG.model}}
    if stream:
        for event in graph.stream(state, config=config):
            print("nodes:", list(event))
        result = graph.get_state(config).values
    else:
        result = graph.invoke(state, config=config)
    final_message = next((m for m in reversed(result["messages"])
                          if isinstance(m, AIMessage) and m.content), None)
    answer = _message_text(final_message) if final_message else ""
    if cacheable and answer:
        semantic_cache.update(cache_prompt, namespace, [Generation(text=answer)])
    run_tree = get_current_run_tree()
    return {
        "answer": answer,
        "state": result,
        "thread_id": thread_id,
        "session_id": session_id,
        "cache_hit": False,
        "langsmith_run_id": str(run_tree.id) if run_tree else None,
    }


In [35]:

def end_session(session_id: str) -> dict[str, Any]:
    staged = stage_session_end(session_id)
    promoted = drain_promotion_queue(session_id=session_id)
    oamp_call(oamp.wait_for_memory_extraction, timeout=300)
    return {"session_id": session_id, "staged": staged, "promoted": promoted}

print("run_agent and end_session are ready.")


run_agent and end_session are ready.


# Part 10 · Deterministic component contract tests

**Yes, this part is necessary.** Live model scenarios test orchestration and integration, but their wording and tool sequence are nondeterministic. These direct tool contracts isolate the model-independent truths the harness must never get wrong.

The assertions prove:

- exactly three inventory rows require new action;
- Berlin ThermaCore M is suppressed by the exact open PO;
- WarmLayer's latest week exceeds its prior average without an internal promotion;
- ThermaCore inventory preserves the expected size coverage;
- UK leads under the governed gross-margin measure;
- institutional and external retrieval return evidence with provenance.

If this part fails, debug data, SQL, or retrieval before inspecting model behavior. Keeping it separate prevents a fluent answer from concealing a broken business contract and prevents model variance from obscuring deterministic defects.

In [36]:
brief_facts = json.loads(morning_brief_inputs.invoke({"excluded_category": ""}))
warm = json.loads(sales_signal.invoke({"product_line": "WarmLayer", "region": "UK"}))
therma = json.loads(inventory_status.invoke({"product_line": "ThermaCore"}))
profit = json.loads(regional_profitability.invoke({}))
institutional = json.loads(institutional_search.invoke({"query": "WarmLayer cold temperature threshold", "limit": 2}))
external = json.loads(external_signal_search.invoke({"region": "UK", "signal_type": "weather"}))

assert len(brief_facts["new_actions"]) == 3
assert [row["PO_ID"] for row in brief_facts["suppressed"]] == ["PO-BER-THC-OPEN"]
assert warm["current_units"] > warm["prior_weekly_average"] and not warm["internal_promotion"]
assert {row["SIZE_CODE"] for row in therma} >= {"XS", "M", "L", "XXL"}
assert profit[0]["REGION"] == "UK" and profit[0]["GROSS_MARGIN"] > profit[1]["GROSS_MARGIN"]
assert institutional and external and external[0]["SOURCE_URL"]
print("PASS: business data, governed metric, suppression, retrieval, and evidence contracts")


PASS: business data, governed metric, suppression, retrieval, and evidence contracts


# Part 11 · Semantic cache in Oracle with `langchain-oracledb`

`OracleSemanticCache` is an application-level boundary **before** LangGraph. It embeds the latest scoped question, searches the Oracle cache table by cosine distance, and either returns a stored `Generation` or lets the normal graph run. A hit skips OAMP context assembly, skill/tool retrieval, Claude, E2B, business tools, and OracleSaver checkpoint writes.

```mermaid
flowchart LR
    Q[Scoped eligible question] --> E[Oracle in-database embedding adapter]
    E --> C{OracleSemanticCache\nCOSINE, threshold 0.25}
    C -->|hit| H[Return stored Generation]
    C -->|miss| G[LangGraph + OracleSaver]
    G --> M[OAMP + semantics + skills + tools + Claude]
    M --> U[Cache update]
    U --> A[Answer]
    H --> A
```

The Oracle integration owns similarity computation through `distance_strategy=DistanceStrategy.COSINE`. The harness therefore does **not** implement a second `cosine_distance` function or calibrate vectors in NumPy. `score_threshold=0.25` is the maximum cosine distance accepted by this configured cache, matching the released integration's documented constructor pattern.

| Constructor argument | Role here |
|---|---|
| `client` | Dedicated Oracle connection for cache operations |
| `embedding` | LangChain adapter around Oracle `VECTOR_EMBEDDING` |
| `table_name` | Explicit application-owned cache table |
| `distance_strategy` | Native integration cosine distance |
| `create_index_if_missing=False` | Keeps physical index creation an explicit DBA decision for this small lab |
| `score_threshold=0.25` | Rejects more distant prompts instead of forcing a nearest hit |

Cache correctness also depends on scope and freshness. The prompt includes tenant and user; the `llm_string` namespace includes model, thinking policy, prompt contract, tenant, user, and a fingerprint of mutable ERPA data. Preference writes, approvals, schedules, explicit mutations, and morning briefs bypass caching. Production systems should replace the teaching fingerprint with governed domain-version invalidation and evaluate the threshold on real query pairs.

Reference: [Oracle semantic cache and chat history with LangChain](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/langchain_ecosystem/langchain_oracle_semantic_cache_chat_history.ipynb).

In [37]:
from langchain_core.embeddings import Embeddings
from langchain_core.outputs import Generation
from langchain_oracledb import OracleSemanticCache
from langchain_oracledb.vectorstores import DistanceStrategy


class OracleInDatabaseEmbeddings(Embeddings):
    """LangChain embedding interface backed by Oracle VECTOR_EMBEDDING."""
    def __init__(self, connection_pool, model_name: str):
        self.pool = connection_pool
        self.model_name = model_name

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        connection = self.pool.acquire()
        try:
            vectors = []
            with connection.cursor() as cursor:
                for text in texts:
                    cursor.execute(
                        f"SELECT VECTOR_EMBEDDING({self.model_name} USING :text AS DATA) FROM dual",
                        {"text": text},
                    )
                    vector = cursor.fetchone()[0]
                    vectors.append(list(vector.tolist()) if hasattr(vector, "tolist") else list(vector))
            return vectors
        finally:
            connection.close()

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]



def scoped_cache_prompt(question: str, user_id: str) -> str:
    return f"tenant={CFG.tenant_id};user={user_id};question={question.strip()}"


def semantic_cache_data_version() -> str:
    row = fetch_all("""SELECT
        (SELECT COUNT(*)||':'||NVL(SUM(on_hand),0)||':'||NVL(SUM(reorder_point),0) FROM erpa_inventory) inventory,
        (SELECT COUNT(*)||':'||NVL(SUM(quantity),0) FROM erpa_purchase_orders WHERE status='OPEN') open_pos,
        (SELECT COUNT(*)||':'||NVL(SUM(units),0) FROM erpa_weekly_sales) sales,
        (SELECT COUNT(*)||':'||NVL(SUM(gross_margin),0) FROM erpa_margin_facts) margins,
        (SELECT COUNT(*)||':'||NVL(TO_CHAR(MAX(observed_at),'YYYYMMDD'),'none') FROM erpa_external_signals) signals
        FROM dual""")[0]
    payload = json.dumps(row, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]


def semantic_cache_namespace(user_id: str) -> str:
    return (f"{CFG.model};thinking=adaptive;erpa-prompt-v2;tenant={CFG.tenant_id};"
            f"user={user_id};data={semantic_cache_data_version()}")


NON_CACHEABLE_MARKERS = (
    "morning brief", "remember", "persist", "do not show", "don't show",
    "stop showing", "exclude", "approve", "send", "draft", "schedule",
    "create", "update", "delete", "write",
)


def is_semantic_cache_eligible(question: str) -> bool:
    lowered = question.casefold()
    return len(lowered.split()) >= 4 and not any(marker in lowered for marker in NON_CACHEABLE_MARKERS)

In [38]:
cache_embeddings = OracleInDatabaseEmbeddings(pool, CFG.embed_model)

CACHE_QUESTION = "What is the current ThermaCore stock position by region and size?"
CACHE_PARAPHRASE = "Show ThermaCore inventory across regions and sizes."
CACHE_UNRELATED = "Why did WarmLayer sales rise in the UK?"

# OracleSemanticCache applies this maximum distance using its configured
# DistanceStrategy.COSINE; no parallel/manual cosine implementation is needed.
CACHE_SCORE_THRESHOLD = 0.25
CACHE_TABLE = "ERPA_SEMANTIC_CACHE"
cache_connection = connect()

In [39]:
OracleSemanticCache.drop_table(cache_connection, CACHE_TABLE)

semantic_cache = OracleSemanticCache(
    client=cache_connection,
    embedding=cache_embeddings,
    table_name=CACHE_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
    create_index_if_missing=False,
    score_threshold=CACHE_SCORE_THRESHOLD,
)
cache_metrics = {"hits": 0, "misses": 0}
cache_events: list[dict[str, Any]] = []

print({
    "table": CACHE_TABLE,
    "embedding_model": CFG.embed_model,
    "dimensions": CFG.embed_dim,
    "distance_strategy": DistanceStrategy.COSINE.name,
    "score_threshold": CACHE_SCORE_THRESHOLD,
})

{'table': 'ERPA_SEMANTIC_CACHE', 'embedding_model': 'ALL_MINILM_L12_V2', 'dimensions': 384, 'distance_strategy': 'COSINE', 'score_threshold': 0.25}


In [40]:
semantic_cache.clear()
cache_metrics.update({"hits": 0, "misses": 0})
cache_events.clear()

before = dict(graph_metrics)
cold_start = time.perf_counter()
cold = run_agent(
    CACHE_QUESTION, session_id="cache-cold-session", thread_id="cache-cold-thread"
)
cold_ms = (time.perf_counter() - cold_start) * 1000
after_cold = dict(graph_metrics)

warm_start = time.perf_counter()
warm = run_agent(
    CACHE_PARAPHRASE, session_id="cache-warm-session", thread_id="cache-warm-thread"
)
warm_ms = (time.perf_counter() - warm_start) * 1000
after_warm = dict(graph_metrics)

assert cold["cache_hit"] is False and warm["cache_hit"] is True
assert after_cold["model_calls"] > before["model_calls"]
assert after_warm == after_cold, "A cache hit must not call the graph model or tools"
assert cold["answer"] == warm["answer"]
print({
    "cold_ms": round(cold_ms), "warm_ms": round(warm_ms),
    "cache_metrics": cache_metrics, "cache_events": cache_events,
    "cold_graph_delta": {k: after_cold[k] - before[k] for k in before},
    "warm_graph_delta": {k: after_warm[k] - after_cold[k] for k in after_cold},
    "same_answer": cold["answer"] == warm["answer"],
})


{'cold_ms': 142413, 'warm_ms': 1767, 'cache_metrics': {'hits': 1, 'misses': 1}, 'cache_events': [{'status': 'MISS', 'question': 'What is the current ThermaCore stock position by region and size?', 'namespace': 'claude-opus-4-8;thinking=adaptive;erpa-prompt-v2;tenant=erpa-workshop;user=alex-course-user;data=3a9ecee9fee2'}, {'status': 'HIT', 'question': 'Show ThermaCore inventory across regions and sizes.', 'namespace': 'claude-opus-4-8;thinking=adaptive;erpa-prompt-v2;tenant=erpa-workshop;user=alex-course-user;data=3a9ecee9fee2'}], 'cold_graph_delta': {'model_calls': 2, 'tool_calls': 2}, 'warm_graph_delta': {'model_calls': 0, 'tool_calls': 0}, 'same_answer': True}


# Part 12 · Live ERPA scenario across five turns

The scenario now exercises the whole custom harness with Claude Opus 4.8. Assertions target grounded state and durable effects rather than brittle exact prose.

| Turn | User intent | Harness seams exercised |
|---|---|---|
| 1 | Prepare a morning brief | semantic meaning, skill/tool discovery, inventory/PO suppression, checkpoint, trace |
| 2 | Explain a WarmLayer spike | time-series tool, institutional vector retrieval, dated external evidence, causal restraint |
| 3 | Inspect ThermaCore sizes | inventory retrieval, E2B generated-code boundary, ScratchFS artifact path |
| 4 | Rank profitability and remember a preference | governed view, explicit durable OAMP preference |
| 5 | Repeat from a fresh session/thread | session-end scratch promotion and cross-session memory recall |

These turns pass `bypass_cache=True` so every intended graph, model, tool, E2B, memory, and checkpoint seam runs live. Part 11 is the isolated semantic-cache acceptance path. The fresh fifth turn is especially important: changing both session and checkpoint thread prevents short-term state from masquerading as durable memory.

The restart test retrieves the durable preference and promoted PO fact with separate focused queries. This avoids one top-k list making unrelated memory obligations compete for rank, especially after a rerunnable workshop has accumulated prior valid records.

In [41]:
SESSION_ID = "erpa-live-session"
THREAD_ID = "erpa-live-thread"

turn1 = run_agent("Prepare my morning brief.", session_id=SESSION_ID, thread_id=THREAD_ID, stream=True, bypass_cache=True)
print(turn1["answer"])
print("LangSmith run:", turn1["langsmith_run_id"])
assert turn1["answer"].strip()


nodes: ['assemble_context']
nodes: ['call_model']
nodes: ['dispatch_tools']
nodes: ['assemble_context']
nodes: ['call_model']
nodes: ['persist']
Morning brief — as of 2026-08-12 (UK & EU · Outerwear & Tops)

New restock actions (below reorder point):
1. **FSH-UK-M** (FieldShell, Outerwear, UK/London) — on-hand 6 vs reorder 32 → gap 26. No open PO.
2. **THC-UK-L** (ThermaCore, Outerwear, UK/London) — on-hand 5 vs reorder 30 → gap 25. No open PO.
3. **PCT-EU-M** (PleatCraft, Tops, EU/Paris) — on-hand 8 vs reorder 35 → gap 27. No open PO.

Suppressed (already covered — not re-raised):
- **THC-EU-BER-M** (ThermaCore, Outerwear, EU/Berlin), on-hand 4 vs 30 — covered by open **PO-BER-THC-OPEN**.

3 new actions need reorder decisions; 1 issue is already covered.
LangSmith run: 019ff61f-92e9-72f1-895a-5f062344323b


In [42]:
turn2 = run_agent(
    "Why did WarmLayer spike in the UK last week? Use internal sales, institutional guidance, and a dated external signal.",
    session_id=SESSION_ID, thread_id=THREAD_ID, bypass_cache=True,
)
print(turn2["answer"])
assert turn2["answer"].strip()


**WarmLayer UK spike — as of 2026-08-12**

**Evidence (internal):**
- Last week's units: **320** vs prior weekly average **100** → **+220%** (`sales_signal`, WarmLayer/UK).
- `promotion_flag` = **N** — no internal promotion was active, so the spike is *not* explained by a markdown.

**Evidence (external, dated):**
- **2026-08-12** UK weather signal: a **cold snap pushed mean temperatures below the 12°C threshold** (source: https://example.test/weather/uk-cold-snap).

**Institutional guidance:**
- Regional merchandising note: *"UK customers respond quickly to temperature changes; present WarmLayer when mean temperatures fall below 12°C"* (oracle://erpa/institutional/regional-merch). Report UK values in GBP.

**Assessment (hypothesis, not proof):**
The +220% lift correlates with a dated UK cold snap crossing the exact 12°C threshold that institutional guidance ties to WarmLayer demand, and there was no offsetting internal promotion. The most likely driver is **weather-led demand**. This 

In [43]:
turn3 = run_agent(
    "Show me ThermaCore stock across regions. Use E2B to analyse or render the size curve and store large output in scratch.",
    session_id=SESSION_ID, thread_id=THREAD_ID, bypass_cache=True,
)
print(turn3["answer"])
assert turn3["answer"].strip()


**ThermaCore stock across regions — as of 2026-08-12** (analysed & rendered via E2B; full report + SVG size curve saved to ScratchFS: `/scratch/thermacore/size_curve.txt`)

**Totals on-hand**
- UK (London): **230** across 4 SKUs
- EU (Paris + Berlin): **216** across 4 SKUs

**Size curve (on-hand by size)**
| Size | UK/London | EU |
|------|-----------|-----|
| XS | 100 (THC-UK-XS) | 70 (THC-EU-PAR-XS) |
| M | 35 (THC-UK-M) | 35 (THC-EU-PAR-M) · **4 Berlin** (THC-EU-BER-M) |
| L | **5 (THC-UK-L)** | 31 (THC-EU-PAR-L) |
| XXL | 90 (THC-UK-XXL) | 80 (THC-EU-PAR-XXL) |

Both regions show a healthy XS/XXL tail but a **thin mid-curve (M/L)** — the core selling sizes are the constrained ones.

**Below reorder point**
- **THC-UK-L** (UK/London, L) — 5 on-hand vs 30 → gap 25, **no open PO → genuine shortfall, needs restock.**
- THC-EU-BER-M (EU/Berlin, M) — 4 on-hand vs 30, but **already covered by open PO-BER-THC-OPEN**, so not re-raised as a new action.

Rendered SVG size-curve chart and the 

In [44]:
turn4 = run_agent(
    "Which regions are most profitable this quarter—and do not show me Accessories in recurring briefs again. Persist that preference.",
    session_id=SESSION_ID, thread_id=THREAD_ID, bypass_cache=True,
)
print(turn4["answer"])

# The persistence node enforces explicit preference capture even if the model answered without calling remember_fact.
if not any("Accessories" in item and any(word in item.lower() for word in ("exclude", "omit", "not show"))
           for item in memory.recall("recurring brief Accessories preference", 8)):
    memory.remember("Exclude Accessories from recurring briefs and reviews.", memory_type="preference")
oamp_call(oamp.wait_for_memory_extraction, timeout=300)
assert any("Accessories" in item for item in memory.recall("recurring brief category exclusion", 8))


**Most profitable regions this quarter** — by governed **gross margin** (not revenue):

| Rank | Region | Gross Margin | GM % | Net Revenue | Avg Discount |
|------|--------|-------------|------|-------------|-------------|
| 1 | **UK** | £48,000 | 38.4% | 125,000 | 5.0% |
| 2 | **US** | 41,000 | 34.75% | 118,000 | 7.0% |
| 3 | **EU** | 36,000 | 27.27% | 132,000 | 12.0% |

**Discounting matters here:** EU actually leads on *revenue* (132,000) and units (1,400), but its **12% average discount** drags gross margin to the bottom (27.27%). On the governed gross-margin definition, **UK is most profitable**, followed by US, then EU. If ranked by revenue the order would flip — so the discount depth is the deciding factor.

**Preference persisted:** Accessories will be excluded from recurring briefs going forward (memory `333354bd`).

Source: `regional_profitability` (ERPA_V_PROFITABILITY). Want me to break the EU discount down by product line to see where the margin is leaking?


In [45]:
session_end = end_session(SESSION_ID)
print(session_end)

fresh_session = "erpa-restart-session"
turn5 = run_agent("Prepare my morning brief.", session_id=fresh_session, thread_id="erpa-restart-thread", bypass_cache=True)
print(turn5["answer"])
restart_preference_memory = memory.recall(
    "Which category should recurring briefs exclude?", 10
)
restart_po_memory = memory.recall(
    "Which purchase order covers Berlin ThermaCore?", 10
)
assert any("Accessories" in item for item in restart_preference_memory)
assert any("PO-BER-THC-OPEN" in item for item in restart_po_memory)
restart_memory = restart_preference_memory + restart_po_memory
print({"fresh_session_preferences": restart_preference_memory[:4],
       "fresh_session_purchase_order_facts": restart_po_memory[:4]})


{'session_id': 'erpa-live-session', 'staged': 0, 'promoted': 0}
# Morning Brief — as of 2026-08-12

Scope: UK & EU · Outerwear and Tops · Accessories excluded.

## Restock actions (new, no covering PO)
1. **THC-UK-L** — ThermaCore, Outerwear, UK/London (L): on-hand **5** vs reorder point **30** → short **25**.
2. **FSH-UK-M** — FieldShell, Outerwear, UK/London (M): on-hand **6** vs reorder point **32** → short **26**.
3. **PCT-EU-M** — PleatCraft, Tops, EU/Paris (M): on-hand **8** vs reorder point **35** → short **27**.

## Suppressed (already covered — no action)
- **THC-EU-BER-M** — ThermaCore, EU/Berlin (M): on-hand 4 vs RP 30, covered by open **PO-BER-THC-OPEN**.

Note: The UK WarmLayer guidance (surface when UK mean temp < 12°C) isn't included — I have no temperature reading to confirm the threshold. Let me know if you'd like me to pull it.
{'fresh_session_preferences': ['Exclude the Accessories category from recurring briefs (morning restock brief and other recurring outputs).', 

# Part 13 · Durable human approval with LangGraph interrupts

A Boolean proposed by the model is not human approval. LangGraph's native human-in-the-loop primitive is `interrupt(payload)`: a node pauses, the configured checkpointer persists its state, and the host later supplies a decision with `Command(resume=...)` under the same `thread_id`.

This part builds a small approval graph on the same OracleSaver rather than using an in-memory approval set. The draft node commits an audit row first; the review node interrupts with a JSON-serializable description; the execute node changes the audit row only after an approved resume command.

```mermaid
stateDiagram-v2
    [*] --> DRAFTED: persist exact payload
    DRAFTED --> PAUSED: interrupt(review request)
    PAUSED --> EXECUTED: Command(resume={approved:true})
    PAUSED --> REJECTED: Command(resume={approved:false})
    EXECUTED --> [*]
    REJECTED --> [*]
```

LangGraph restarts an interrupted node from its beginning when resumed. Therefore side effects before `interrupt()` must be idempotent, or—as here—placed in a preceding node. The adapter remains a demonstration and makes no external Notion/email mutation; the durable pause, resume, and audit transitions are real.

Reference: [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts).

In [46]:
from langgraph.types import Command, interrupt


class ApprovalState(TypedDict, total=False):
    action_id: str
    action_type: str
    payload: dict[str, Any]
    status: str
    review: dict[str, Any]
    adapter_result: dict[str, Any]


def persist_action_draft(state: ApprovalState) -> ApprovalState:
    execute("""MERGE INTO erpa_action_audit d
      USING (SELECT :action_id action_id FROM dual) s ON(d.action_id=s.action_id)
      WHEN NOT MATCHED THEN INSERT(action_id,action_type,payload,status)
        VALUES(s.action_id,:action_type,:payload,'DRAFTED')""",
        {"action_id": state["action_id"], "action_type": state["action_type"],
         "payload": json.dumps(state["payload"])})
    return {"status": "DRAFTED"}


def request_human_review(state: ApprovalState) -> ApprovalState:
    decision = interrupt({
        "kind": "action_approval",
        "action_id": state["action_id"],
        "action_type": state["action_type"],
        "payload": state["payload"],
        "instruction": "Approve or reject this exact drafted action.",
    })
    approved = bool(decision.get("approved"))
    return {
        "review": dict(decision),
        "status": "APPROVED" if approved else "REJECTED",
    }


def apply_reviewed_action(state: ApprovalState) -> ApprovalState:
    if state["status"] != "APPROVED":
        execute("UPDATE erpa_action_audit SET status='REJECTED' WHERE action_id=:action_id",
                {"action_id": state["action_id"]})
        return {"adapter_result": {"status": "rejected", "side_effect": False}}
    execute("""UPDATE erpa_action_audit SET status='EXECUTED',
               approved_at=SYSTIMESTAMP,executed_at=SYSTIMESTAMP
               WHERE action_id=:action_id AND status='DRAFTED'""",
            {"action_id": state["action_id"]})
    return {"status": "EXECUTED", "adapter_result": {
        "status": "executed", "adapter": "demonstration", "side_effect": True,
    }}


approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("persist_draft", persist_action_draft)
approval_builder.add_node("human_review", request_human_review)
approval_builder.add_node("apply_action", apply_reviewed_action)
approval_builder.add_edge(START, "persist_draft")
approval_builder.add_edge("persist_draft", "human_review")
approval_builder.add_edge("human_review", "apply_action")
approval_builder.add_edge("apply_action", END)
approval_graph = approval_builder.compile(checkpointer=checkpointer)

approval_action_id = uuid.uuid4().hex
approval_thread_id = f"approval-{approval_action_id}"
approval_config = {"configurable": {"thread_id": approval_thread_id}}
approval_input: ApprovalState = {
    "action_id": approval_action_id,
    "action_type": "notion_update",
    "payload": {"title": "ERPA morning brief", "body": turn1["answer"]},
}

approval_paused = approval_graph.invoke(approval_input, config=approval_config)
approval_interrupts = approval_paused.get("__interrupt__", ())
approval_before = fetch_all("SELECT action_id,status,approved_at,executed_at FROM erpa_action_audit WHERE action_id=:i",
                            {"i": approval_action_id})[0]
assert approval_interrupts and approval_before["STATUS"] == "DRAFTED"

# This dictionary represents the deliberate decision returned by a notebook/UI host.
approval_resumed = approval_graph.invoke(
    Command(resume={"approved": True, "reviewer": "course-human"}),
    config=approval_config,
)
approval_after = fetch_all("SELECT action_id,status,approved_at,executed_at FROM erpa_action_audit WHERE action_id=:i",
                           {"i": approval_action_id})[0]
assert approval_resumed["status"] == "EXECUTED"
assert approval_after["STATUS"] == "EXECUTED" and approval_after["APPROVED_AT"] is not None
display(pd.DataFrame([approval_before, approval_after], index=["paused", "resumed"]))
print({"interrupt_count": len(approval_interrupts), "thread_id": approval_thread_id,
       "before": approval_before["STATUS"], "after": approval_after["STATUS"]})

,ACTION_ID,STATUS,APPROVED_AT,EXECUTED_AT
paused,895a482a9a5e48a6aaa4bd3002e943d8,DRAFTED,NaT,NaT
resumed,895a482a9a5e48a6aaa4bd3002e943d8,EXECUTED,2026-08-12 13:23:24.437980,2026-08-12 13:23:24.437980


{'interrupt_count': 1, 'thread_id': 'approval-895a482a9a5e48a6aaa4bd3002e943d8', 'before': 'DRAFTED', 'after': 'EXECUTED'}


# Part 14 · Oracle Scheduler as the cron boundary

`DBMS_SCHEDULER` persists the weekday 08:00 cadence inside Oracle. Its PL/SQL job does not attempt to run Python or call Claude from the database; it inserts a durable request into `ERPA_AUTOMATION_QUEUE`. A Python worker claims that request and sends it through the same `run_agent` boundary, LangGraph, tools, memory, cache policy, OracleSaver, and LangSmith tracing used interactively.

This queue pattern separates reliable timing from model execution. Oracle owns calendar state and enqueue durability; the application worker owns credentials, network calls, retries, and graph execution. The result preview and LangSmith run ID are written back to the same request row.

The cell forces one job run with `use_current_session=>TRUE` so the notebook can test it immediately, then processes the oldest queued request. A production worker would add atomic claim leases, attempts, backoff, dead-letter state, and idempotent delivery.

In [47]:
ddl("""CREATE TABLE erpa_automation_queue (
    request_id VARCHAR2(80) PRIMARY KEY, user_id VARCHAR2(80), prompt VARCHAR2(1000),
    status VARCHAR2(20) DEFAULT 'QUEUED', queued_at TIMESTAMP DEFAULT SYSTIMESTAMP,
    completed_at TIMESTAMP, result_preview VARCHAR2(2000), langsmith_run_id VARCHAR2(80))""")

create_or_replace_job(
    "ERPA_MORNING_BRIEF_JOB",
    """BEGIN
       INSERT INTO erpa_automation_queue(request_id,user_id,prompt)
       VALUES(RAWTOHEX(SYS_GUID()),'alex-course-user','Morning brief.');
       COMMIT;
     END;""",
    "FREQ=WEEKLY;BYDAY=MON,TUE,WED,THU,FRI;BYHOUR=8;BYMINUTE=0;BYSECOND=0",
)
execute("BEGIN DBMS_SCHEDULER.RUN_JOB('ERPA_MORNING_BRIEF_JOB',use_current_session=>TRUE); END;")
queued = fetch_all("""SELECT request_id,user_id,prompt FROM erpa_automation_queue
                      WHERE status='QUEUED' ORDER BY queued_at FETCH FIRST 1 ROW ONLY""")
assert queued

request = queued[0]
scheduled = run_agent(
    request["PROMPT"], session_id=f"scheduled-{request['REQUEST_ID'][:12]}",
    thread_id=f"scheduled-{request['REQUEST_ID']}", user_id=request["USER_ID"],
)
execute("""UPDATE erpa_automation_queue SET status='COMPLETED',completed_at=SYSTIMESTAMP,
           result_preview=:preview,langsmith_run_id=:run_id WHERE request_id=:request_id""",
        {"preview": scheduled["answer"][:1900], "run_id": scheduled["langsmith_run_id"],
         "request_id": request["REQUEST_ID"]})
print(fetch_all("SELECT request_id,status,langsmith_run_id FROM erpa_automation_queue WHERE request_id=:i",
                {"i": request["REQUEST_ID"]})[0])


{'REQUEST_ID': '58DA7E40F3A813C2E063030011AC2B4B', 'STATUS': 'COMPLETED', 'LANGSMITH_RUN_ID': '019ff624-6ef1-7751-9c9d-88c42047c918'}


# Part 15 · Checkpoints, traces, restart evidence, and final acceptance

A checkpoint, memory, and trace answer different operational questions:

| Artifact | Question answered | Owner |
|---|---|---|
| OracleSaver checkpoint | “What graph state can resume under this thread?” | LangGraph Oracle integration |
| OAMP memory | “What should ERPA recall in another conversation?” | Oracle Agent Memory |
| LangSmith trace | “Which nodes, tools, retrievals, timings, and model calls happened?” | LangSmith |
| Oracle audit/queue row | “What business transition or scheduled request was committed?” | Application schema |

The cell reads the main graph checkpoint, then creates a fresh Oracle pool and saver and reads it again. That proves restart visibility rather than accidentally reading an in-memory object. LangSmith ingestion is eventually consistent, so trace verification polls for a bounded interval. It reports `VISIBLE` or `UNVERIFIED` with the real authorization, workspace-routing, rate-limit, or ingestion error using the current asynchronous `runs.query` API. Set `ERPA_REQUIRE_LANGSMITH_VISIBLE=1` in CI when external trace visibility must be a hard acceptance gate; this keeps invalid or missing SaaS workspace routing from being confused with a failure of the Oracle harness itself.

The integrated acceptance cell that follows checks the complete architecture in one place. It is kept in this part instead of a separate “Part 16”: final acceptance is evidence for checkpointing/observability, not another harness component.

In [48]:
checkpoint_config = {"configurable": {"thread_id": THREAD_ID}}
checkpoint = checkpointer.get(checkpoint_config)
assert checkpoint is not None

fresh_pool = oracledb.create_pool(
    user=CFG.oracle_user, password=CFG.oracle_password, dsn=CFG.oracle_dsn,
    min=1, max=2, increment=1,
)
fresh_saver = OracleSaver(fresh_pool, json_size_threshold_mb=0.0)
fresh_checkpoint = fresh_saver.get(checkpoint_config)
assert fresh_checkpoint is not None
fresh_pool.close(force=True)

from langsmith import Client

langsmith_workspace_id = os.getenv("LANGSMITH_WORKSPACE_ID", "").strip() or None
ls_client = Client(api_key=LANGSMITH_API_KEY, workspace_id=langsmith_workspace_id)
trace_ids = [turn1["langsmith_run_id"], turn2["langsmith_run_id"],
             turn3["langsmith_run_id"], turn4["langsmith_run_id"]]
trace_ids = [item for item in trace_ids if item]
assert trace_ids, "The traced run boundary did not produce local LangSmith run IDs"

# Current LangSmith SDKs expose query through the generated async runs client.
# Query all expected IDs in one request and poll only for eventual consistency.
visible = []
trace_verification_error = ""
deadline = time.monotonic() + 60
while time.monotonic() < deadline and len(visible) < len(trace_ids):
    try:
        page = await ls_client.runs.query(ids=trace_ids, page_size=len(trace_ids))
        visible = list(page.items)
        trace_verification_error = ""
    except Exception as exc:
        trace_verification_error = f"{type(exc).__name__}: {str(exc)[:300]}"
        # Authorization, workspace routing, and rate-limit failures will not be
        # repaired by polling this notebook run. Surface them immediately.
        if type(exc).__name__ in {
            "AuthenticationError", "PermissionDeniedError", "RateLimitError",
            "LangSmithAuthError", "LangSmithRateLimitError",
        }:
            break
    if len(visible) < len(trace_ids):
        time.sleep(2)

langsmith_trace_status = "VISIBLE" if visible else "UNVERIFIED"
if langsmith_trace_status != "VISIBLE":
    print("LANGSMITH STATUS UNVERIFIED:", trace_verification_error or
          "No run became visible before the bounded deadline")
    print("Set LANGSMITH_WORKSPACE_ID when the API key is scoped to multiple workspaces.")
    if os.getenv("ERPA_REQUIRE_LANGSMITH_VISIBLE", "0") == "1":
        raise RuntimeError("LangSmith trace visibility is required but could not be verified")

print({"oracle_checkpoint": True, "fresh_saver_read": True,
       "langsmith_status": langsmith_trace_status,
       "langsmith_runs_visible": len(visible), "trace_ids": trace_ids})

LANGSMITH STATUS UNVERIFIED: BadRequestError: Error code: 400 - {'type': 'about:blank', 'title': 'Bad Request', 'status': 400, 'detail': 'invalid request: set project_ids or reference_dataset_id', 'instance': '/v2/runs/query'}
Set LANGSMITH_WORKSPACE_ID when the API key is scoped to multiple workspaces.
{'oracle_checkpoint': True, 'fresh_saver_read': True, 'langsmith_status': 'UNVERIFIED', 'langsmith_runs_visible': 0, 'trace_ids': ['019ff61f-92e9-72f1-895a-5f062344323b', '019ff620-5e1b-7723-83bb-d7a1390a97de', '019ff621-6958-7b90-9ecc-67273d50ade4', '019ff622-b667-7961-ba4e-8d91ad99adf8']}


## Integrated final acceptance evidence

There is no separate Part 16. The previous version's final acceptance section was an extra number for a verification step, not an architectural layer. It now concludes Part 15 and checks concrete objects and outcomes:

- required ERPA tables and Scheduler job exist in Oracle;
- the main graph is recoverable through OracleSaver;
- deterministic inventory, PO-suppression, and profitability contracts hold;
- fresh-session OAMP recall contains both an explicit preference and promoted scratch fact;
- displayed promotion receipts correspond to new database memory records;
- E2B is the generated-code provider;
- semantic cache proves both miss and hit paths; and
- LangGraph HITL proves a durable interrupt followed by an approved Oracle audit transition; and
- LangSmith local run IDs exist, while server visibility is reported separately as `VISIBLE` or `UNVERIFIED` (and can be made strict in CI).

Passing this cell means the notebook's stated seams executed together. It is not a substitute for load, security, retrieval-quality, or failure-injection testing.

In [49]:
required_tables = {
    "ERPA_PRODUCTS", "ERPA_INVENTORY", "ERPA_SCRATCH_FILES",
    "ERPA_MEMORY_PROMOTION_QUEUE", "ERPA_SEMANTIC_CATALOG",
    "ERPA_TOOL_REGISTRY", "ERPA_SKILL_REGISTRY", "ERPA_AUTOMATION_QUEUE",
    "ERPA_SEMANTIC_CACHE", "ERPA_ACTION_AUDIT",
}
actual_tables = {row["TABLE_NAME"] for row in fetch_all(
    "SELECT table_name FROM user_tables WHERE table_name LIKE 'ERPA_%'"
)}
assert required_tables <= actual_tables
assert fetch_all("SELECT COUNT(*) n FROM user_scheduler_jobs WHERE job_name='ERPA_MORNING_BRIEF_JOB'")[0]["N"] == 1
assert checkpointer.get({"configurable": {"thread_id": THREAD_ID}}) is not None
assert len(brief_facts["new_actions"]) == 3
assert brief_facts["suppressed"][0]["PO_ID"] == "PO-BER-THC-OPEN"
assert profit[0]["REGION"] == "UK"
assert any("Accessories" in item for item in restart_memory)
assert any("PO-BER-THC-OPEN" in item for item in restart_memory)
assert promoted_db_entries and all(item["memory_id"].startswith("scratch-") for item in promoted_db_entries)
assert all(item["record_type"] == "fact" for item in promoted_db_entries)
assert smoke["provider"] == "e2b"
assert cache_metrics["hits"] >= 1 and cache_metrics["misses"] >= 1
assert approval_interrupts and approval_after["STATUS"] == "EXECUTED"
assert approval_graph.get_state(approval_config).values["status"] == "EXECUTED"
assert trace_ids

print("PASS: ERPA runtime contracts completed; external observability status reported separately")
print({
    "database": version_row,
    "model": CFG.model,
    "thinking": "adaptive",
    "memory": f"Oracle Agent Memory {OAMP_VERSION}",
    "checkpointer": "OracleSaver",
    "semantic_cache": "OracleSemanticCache / DistanceStrategy.COSINE / 0.25",
    "human_in_the_loop": "LangGraph interrupt + Command(resume) + Oracle audit",
    "sandbox": "E2B",
    "tracing": {"project": CFG.langsmith_project, "server_status": langsmith_trace_status},
    "business_contract": "3 actions, 1 PO suppression, UK gross-margin leader",
})

PASS: ERPA runtime contracts completed; external observability status reported separately
{'database': {'PRODUCT': 'Oracle AI Database 26ai Free', 'VERSION_FULL': '23.26.0.0.0'}, 'model': 'claude-opus-4-8', 'thinking': 'adaptive', 'memory': 'Oracle Agent Memory 26.6.0', 'checkpointer': 'OracleSaver', 'semantic_cache': 'OracleSemanticCache / DistanceStrategy.COSINE / 0.25', 'human_in_the_loop': 'LangGraph interrupt + Command(resume) + Oracle audit', 'sandbox': 'E2B', 'tracing': {'project': 'erpa-custom-harness', 'server_status': 'UNVERIFIED'}, 'business_contract': '3 actions, 1 PO suppression, UK gross-margin leader'}


# Optional scoped cleanup

There is no separate Part 16 for cleanup either: cleanup is a lifecycle option, not a harness capability. The default retains schema data, memory, checkpoints, scratch artifacts, audit evidence, cache rows, and Scheduler jobs for inspection.

Set `ERPA_KEEP_DATA=0` before running to enable the explicit cleanup cell. It removes only the named Scheduler jobs and semantic-cache table created by this notebook, then closes clients, executors, pools, and connections. It never drops the application schema, deletes the Oracle container, removes its volume, or erases LangSmith traces automatically.

Production teardown should follow retention, legal-hold, backup, and tenant-deletion policy rather than a notebook flag.

In [50]:
if CFG.keep_data:
    print("ERPA_KEEP_DATA=1: workshop data retained for inspection.")
else:
    for job in ("ERPA_MORNING_BRIEF_JOB", "ERPA_SEMANTIC_REFRESH_JOB"):
        try:
            execute("BEGIN DBMS_SCHEDULER.DROP_JOB(:name,force=>TRUE); END;", {"name": job})
        except oracledb.DatabaseError:
            pass
    OracleSemanticCache.drop_table(cache_connection, CACHE_TABLE)
    print("Scheduler jobs and semantic-cache table removed. Set ERPA_KEEP_DATA=1 for normal workshop runs.")

oamp_call(oamp.close, timeout=120)
OAMP_EXECUTOR.shutdown(wait=True)
checkpoint_connection.close()
cache_connection.close()
pool.close(force=True)


ERPA_KEEP_DATA=1: workshop data retained for inspection.


## What the learner should now be able to explain

- Why an agent harness is a governed runtime rather than a larger model prompt.
- How Oracle business rows, semantic definitions, ScratchFS files, graph checkpoints, OAMP memory, cache entries, and traces differ in purpose and lifetime.
- Why the semantic layer has both a retrieval path (comments/hints/workload → vector catalog) and an enforcement path (governed view/tool → source rows).
- Why a logical session-end trigger stages promotion and a Python OAMP consumer completes it.
- How the displayed queue and OAMP records prove scratch content became new durable database memory.
- Why tool discovery, tool binding, allowlisted dispatch, and E2B isolation are separate controls.
- Why direct deterministic tool contracts remain necessary beside live model scenarios.
- How Oracle's `OracleSemanticCache` applies `DistanceStrategy.COSINE` and a threshold without a parallel NumPy distance function.
- Why cache scope, eligibility, namespace versioning, and data invalidation matter as much as similarity.
- How LangGraph `interrupt()` plus `Command(resume=...)` creates a durable approval boundary with OracleSaver.
- How Oracle Scheduler, LangGraph, OracleSaver, OAMP, and LangSmith make the assistant proactive, resumable, stateful, and observable.
- Where Anthropic prompt caching would fit, and why it is complementary to—not a replacement for—semantic caching or memory.

The remaining component-map gaps are deliberate design work for a larger system: dedicated entity memory APIs, broad tool-result compaction, live external connectors, MCP transport, multi-agent delegation, persona lifecycle, self-modification controls, and richer context telemetry.